# Panorama Bois


In [1]:
%pip install requests beautifulsoup4 pandas numpy folium

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\ASUS\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


## Imports les packages et definir les fonctions

In [2]:
import csv
import html
import json
import re
import threading
import time
import unicodedata
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from urllib.parse import urljoin

import folium
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from bs4.element import NavigableString, Tag
from folium.plugins import MarkerCluster
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


# 1. Fonctions utilitaires communes

def est_vide(valeur):
    if valeur is None:
        return True

    try:
        if pd.isna(valeur):
            return True
    except (TypeError, ValueError):
        pass

    texte = str(valeur).strip().lower()
    texte_normalise = re.sub(r"\s+", " ", texte)

    valeurs_vides = {
        "",
        "nan",
        "none",
        "null",
        "<na>",
        "-",
        "--",
        "x",
        "xx",
        "x x",
    }

    return texte_normalise in valeurs_vides


def nettoyer_texte(element):
    if est_vide(element):
        return ""

    if hasattr(element, "get_text"):
        texte = element.get_text(" ", strip=True)
    else:
        texte = str(element)

    texte = texte.replace("\xa0", " ")
    return re.sub(r"\s+", " ", texte).strip()


def normaliser_code_postal(valeur):
    if est_vide(valeur):
        return ""

    code_postal = nettoyer_texte(valeur)

    if re.fullmatch(r"\d+\.0", code_postal):
        code_postal = code_postal[:-2]

    if code_postal.isdigit() and len(code_postal) < 5:
        code_postal = code_postal.zfill(5)

    return code_postal


def normaliser_chaine(valeur):
    """Minuscules, sans accent ni ponctuation : utilise pour comparer des noms."""

    texte = nettoyer_texte(valeur).casefold()
    texte = unicodedata.normalize("NFD", texte)
    texte = "".join(car for car in texte if unicodedata.category(car) != "Mn")
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9]+", " ", texte)).strip()


def normaliser_nom_commune(valeur):
    """Développer uniquement les abréviations ST et STE comme mots entiers."""
    mots = normaliser_chaine(valeur).split()
    abreviations = {"st": "saint", "ste": "sainte"}
    return " ".join(abreviations.get(mot, mot) for mot in mots)


def communes_equivalentes(commune_a, commune_b):
    nom_a = normaliser_nom_commune(commune_a)
    nom_b = normaliser_nom_commune(commune_b)
    return bool(nom_a and nom_b and nom_a == nom_b)


# 2. Cache des pages projets, partage par les extractions generales et acteurs

pages_projets_cache = {}


def charger_page_projet(url, timeout=20):
    if url not in pages_projets_cache:
        response = requests.get(
            url, headers={"User-Agent": "Mozilla/5.0"}, timeout=timeout
        )
        response.raise_for_status()
        pages_projets_cache[url] = response.text

    contenu_html = pages_projets_cache[url]
    return BeautifulSoup(contenu_html, "html.parser"), contenu_html


# 3. Session HTTP resiliente et appels aux API geographiques
#    Utilisee par l'extraction des projets (2.1) et par le geocodage (2.2, 3.3).

session_geographique = requests.Session()
session_geographique.mount(
    "https://",
    HTTPAdapter(
        max_retries=Retry(
            total=4,
            connect=4,
            read=4,
            backoff_factor=0.5,
            status_forcelist=[429, 500, 502, 503, 504],
            allowed_methods=["GET"],
            respect_retry_after_header=True,
        ),
        pool_maxsize=16,
    ),
)

URL_API_COMMUNES = "https://geo.api.gouv.fr/communes"
CHAMPS_COMMUNE = "nom,code,centre,codesPostaux,departement,region"

verrou_api = threading.Lock()
cache_api = {}
erreurs_api = []


def appeler_api_json(url, params, timeout=20):
    """Appel GET mis en cache.

    Retourne les donnees, ou None en cas d'echec reseau. Un echec n'est jamais
    mis en cache : il reste distinguable d'un resultat vide et peut etre rejoue.
    """

    cle = (url, tuple(sorted(params.items())))

    with verrou_api:
        if cle in cache_api:
            return cache_api[cle]

    try:
        reponse = session_geographique.get(url, params=params, timeout=timeout)
        reponse.raise_for_status()
        donnees = reponse.json()
    except Exception as erreur:
        with verrou_api:
            erreurs_api.append((url, dict(params), type(erreur).__name__))
        return None

    with verrou_api:
        cache_api[cle] = donnees
    return donnees


def chercher_communes(**params):
    """Interroge geo.api.gouv.fr/communes. Retourne (liste de communes, reseau_ok)."""

    donnees = appeler_api_json(
        URL_API_COMMUNES, {"fields": CHAMPS_COMMUNE, "format": "json", **params}
    )
    return (donnees or [], donnees is not None)


def communes_exactes(candidats, nom_commune):
    """Toutes les communes dont le nom officiel correspond exactement.

    Plusieurs communes peuvent porter le meme nom : Arcon (42370, Loire) et
    Arcon (25300, Doubs). La liste complete est donc renvoyee, la levee
    d'ambiguite se fait ensuite avec le code postal source.
    """

    return [
        candidat
        for candidat in candidats or []
        if communes_equivalentes(candidat.get("nom", ""), nom_commune)
    ]


def commune_exacte(candidats, nom_commune):
    correspondances = communes_exactes(candidats, nom_commune)
    return correspondances[0] if len(correspondances) == 1 else None


def codes_postaux_officiels(commune_info):
    return [
        normaliser_code_postal(code)
        for code in (commune_info or {}).get("codesPostaux", []) or []
        if normaliser_code_postal(code)
    ]


def departement_depuis_code_postal(code_postal):
    """Code departement deduit du code postal ('42370' -> '42', '97400' -> '974').

    La Corse est exclue : 20xxx ne permet pas de distinguer 2A de 2B.
    """

    code_postal = normaliser_code_postal(code_postal)
    if len(code_postal) != 5 or code_postal.startswith("20"):
        return ""
    return code_postal[:3] if code_postal[:2] in {"97", "98"} else code_postal[:2]


def resoudre_commune_par_nom(commune_source, code_postal_source=""):
    """Commune officielle portant exactement ce nom, ambiguites levees.

    Retourne (commune, reseau_ok, homonymes_non_resolus).
    """

    candidats, reseau_ok = chercher_communes(nom=commune_source)
    homonymes = communes_exactes(candidats, commune_source)

    if len(homonymes) <= 1:
        return (homonymes[0] if homonymes else None), reseau_ok, False

    # Plusieurs communes homonymes : le code postal source departage.
    # Cas reel : "Les grands Murcins 42370 Arcon" ne doit pas devenir Arcon (25300).
    code_postal_source = normaliser_code_postal(code_postal_source)
    par_code_postal = [
        commune for commune in homonymes
        if code_postal_source in codes_postaux_officiels(commune)
    ]
    if len(par_code_postal) == 1:
        return par_code_postal[0], reseau_ok, False

    departement_source = departement_depuis_code_postal(code_postal_source)
    par_departement = [
        commune for commune in homonymes
        if departement_source
        and commune.get("departement", {}).get("code", "") == departement_source
    ]
    if len(par_departement) == 1:
        return par_departement[0], reseau_ok, False

    # Aucun element ne departage : les valeurs de la page sont conservees.
    return None, reseau_ok, True


def resoudre_localisation_administrative(code_postal_source="", commune_source=""):
    """Commune officielle a partir du code postal et du nom lus sur la page.

    Retourne (commune officielle ou None, code postal retenu, statut).
    """

    code_postal_source = normaliser_code_postal(code_postal_source)
    commune_source = nettoyer_texte(commune_source)

    candidats_cp, reseau_ok = (
        chercher_communes(codePostal=code_postal_source) if code_postal_source else ([], True)
    )

    # a. Le code postal et le nom de la page concordent.
    correspondance = commune_exacte(candidats_cp, commune_source)
    if correspondance is not None:
        return correspondance, code_postal_source, "postal_et_commune_confirmes"

    # b. Un nom officiel exact peut corriger un code postal source errone.
    #    Exemple : Le Grand-Bornand, ecrit 73470 sur la page, est en 74450.
    homonymes_non_resolus = False
    if commune_source:
        correspondance, reseau_nom, homonymes_non_resolus = resoudre_commune_par_nom(
            commune_source, code_postal_source
        )
        reseau_ok = reseau_ok and reseau_nom

        if correspondance is not None:
            codes_officiels = codes_postaux_officiels(correspondance)
            if code_postal_source in codes_officiels or not codes_officiels:
                return correspondance, code_postal_source, "commune_confirmee"
            return correspondance, codes_officiels[0], "commune_confirmee_code_postal_corrige"

    # c. Un code postal qui ne designe qu'une commune est un dernier recours
    #    deterministe. C'est le cas de 74190, qui ramene le SITOM a Passy.
    if len(candidats_cp) == 1:
        return candidats_cp[0], code_postal_source, "commune_corrigee_par_postal_unique"

    if not reseau_ok:
        statut = "erreur_reseau"
    elif homonymes_non_resolus:
        statut = "commune_homonyme_non_resolue"
    elif not code_postal_source and not commune_source:
        statut = "administration_absente"
    elif not code_postal_source:
        statut = "commune_introuvable"
    elif not candidats_cp:
        statut = "code_postal_introuvable"
    elif len(candidats_cp) > 1:
        statut = "postal_partage_non_resolu"
    else:
        statut = "administration_non_resolue"

    return None, code_postal_source, statut


def resumer_erreurs_api(etape=""):
    """Resume compact des erreurs reseau, au lieu d'une ligne par erreur."""

    if not erreurs_api:
        print(f"Appels API {etape}: aucune erreur reseau ({len(cache_api)} reponses en cache)")
        return

    compteur = {}
    for _, _, type_erreur in erreurs_api:
        compteur[type_erreur] = compteur.get(type_erreur, 0) + 1

    detail = ", ".join(f"{nom} x{nombre}" for nom, nombre in sorted(compteur.items()))
    print(f"Appels API {etape}: {len(erreurs_api)} echecs reseau rejoues ({detail})")
    erreurs_api.clear()


# 1. Collecte des liens

In [3]:
headers = {"User-Agent": "Mozilla/5.0"}

# peut changer a la region et mise a jours
base_url = "https://www.panoramabois.fr/projets/?regions%5B0%5D=1&orderBy=date_desc&participations%5B0%5D=14&page={page}"

# Lecture de la premiere page pour detecter automatiquement le nombre total de pages
response = requests.get(base_url.format(page=1), headers=headers)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")
pages = []

for a in soup.select(".Pagination a[href*='page=']"):
    href = a.get("href", "")
    match = re.search(r"page=(\d+)", href)
    if match:
        pages.append(int(match.group(1)))

max_page = max(pages) if pages else 1
print("Nombre total de pages :", max_page)

url_all = [
    base_url.format(page=page)
    for page in range(1, max_page + 1)
]

projets_ids = set()

for url in url_all:
    print("Reading:", url)

    response = requests.get(url, headers=headers)
    response.raise_for_status()

    html = response.text

    ids_trouves = re.findall(r"/projets/(\d+)", html)

    projets_ids.update(ids_trouves)

print(projets_ids)
print("nb_projets:", len(projets_ids))

# Sauvegarde des liens projets pour les étapes suivantes
liens_projets = [
    f"https://www.panoramabois.fr/projets/{projet_id}"
    for projet_id in sorted(projets_ids)
]

df_liens_projets = pd.DataFrame({"url_projet": liens_projets})
df_liens_projets.to_csv("liens_projets.csv", index=False, encoding="utf-8-sig")

print("Liens projets enregistrés dans : liens_projets.csv")


Nombre total de pages : 31
Reading: https://www.panoramabois.fr/projets/?regions%5B0%5D=1&orderBy=date_desc&participations%5B0%5D=14&page=1
Reading: https://www.panoramabois.fr/projets/?regions%5B0%5D=1&orderBy=date_desc&participations%5B0%5D=14&page=2
Reading: https://www.panoramabois.fr/projets/?regions%5B0%5D=1&orderBy=date_desc&participations%5B0%5D=14&page=3
Reading: https://www.panoramabois.fr/projets/?regions%5B0%5D=1&orderBy=date_desc&participations%5B0%5D=14&page=4
Reading: https://www.panoramabois.fr/projets/?regions%5B0%5D=1&orderBy=date_desc&participations%5B0%5D=14&page=5
Reading: https://www.panoramabois.fr/projets/?regions%5B0%5D=1&orderBy=date_desc&participations%5B0%5D=14&page=6
Reading: https://www.panoramabois.fr/projets/?regions%5B0%5D=1&orderBy=date_desc&participations%5B0%5D=14&page=7
Reading: https://www.panoramabois.fr/projets/?regions%5B0%5D=1&orderBy=date_desc&participations%5B0%5D=14&page=8
Reading: https://www.panoramabois.fr/projets/?regions%5B0%5D=1&orderB

# 2.1 Extraction des informations générales

In [4]:
base_site = "https://www.panoramabois.fr/projets"

projets_data = []
projets_erreurs = []


df_liens_projets = pd.read_csv("liens_projets.csv", dtype=str)
liens_projets = df_liens_projets["url_projet"].dropna().astype(str).str.strip()
liens_projets = liens_projets[liens_projets != ""].drop_duplicates().tolist()



# 0. Définir les fonctions


def extraire_localisation_projet(soup):
    code_postal = ""
    commune = ""
    departement = ""
    region = ""

    div_address = soup.find("div", class_="Project-address")

    if div_address:
        texte = nettoyer_texte(div_address)

        # Exemple :
        # 38090 Vaulx-Milieu - Isere - Auvergne-Rhone-Alpes
        parties = [partie.strip() for partie in re.split(r"\s+-\s+", texte)]
        partie_cp_commune = parties[0] if parties else ""

        match = re.match(r"^(\d{5})\s*(.*)$", partie_cp_commune)

        if match:
            code_postal = match.group(1).strip()
            commune = match.group(2).strip()

        if len(parties) > 1:
            departement = parties[1]

        if len(parties) > 2:
            region = parties[2]

    return code_postal, commune, departement, region

def extraire_infos_tableau(soup):
    infos = {
        "categorie": "",
        "numero_projet": "",
        "adresse_projet": "",
        "date_candidature": "",
    }

    tab1 = soup.select_one("div#tab-1")

    if tab1:
        for tr in tab1.select("tr"):
            th = tr.find("th")
            td = tr.find("td")

            if not th or not td:
                continue

            cle = nettoyer_texte(th)
            valeur = nettoyer_texte(td)

            if cle == "Catégorie":
                infos["categorie"] = valeur

            elif cle == "Numéro de projet":
                infos["numero_projet"] = valeur

            elif cle == "Adresse du projet":
                infos["adresse_projet"] = valeur

            elif cle == "Date de candidature":
                infos["date_candidature"] = valeur

    return infos


def extraire_coordonnees(soup, html):
    latitude = ""
    longitude = ""

    # On cherche d'abord dans div#tab-4
    tab4 = soup.select_one("div#tab-4")

    if tab4:
        texte_recherche = str(tab4)
    else:
        texte_recherche = html

    pattern_coords = r"position\s*:\s*new\s+google\.maps\.LatLng\s*\(\s*([+-]?\d+(?:\.\d+)?)\s*,\s*([+-]?\d+(?:\.\d+)?)\s*\)"
    match_coords = re.search(pattern_coords, texte_recherche)

    # Si on ne trouve pas dans tab-4, on cherche dans toute la page HTML
    if not match_coords:
        match_coords = re.search(pattern_coords, html)

    if match_coords:
        latitude = match_coords.group(1)
        longitude = match_coords.group(2)

    return latitude, longitude


# 1. Boucle principale


for position, lien in enumerate(liens_projets, start=1):

    match_id_projet = re.search(r"/projets/(\d+)", lien)
    projet_id_url = match_id_projet.group(1) if match_id_projet else ""

    if position % 25 == 0 or position == len(liens_projets):
        print(f"Lecture des projets : {position}/{len(liens_projets)}", end="\r")

    try:
        soup, html = charger_page_projet(lien)

        # 1.1 Informations générales

        titre = soup.select_one("h1.Project-title")
        nom_projet = nettoyer_texte(titre)

        cat_concours = soup.select_one("div.Project-category")
        categorie_concours = nettoyer_texte(cat_concours)

        code_postal, commune, departement, region = extraire_localisation_projet(soup)

        infos_tableau = extraire_infos_tableau(soup)

        categorie = infos_tableau["categorie"]
        numero_projet = infos_tableau["numero_projet"]
        adresse_precise = infos_tableau["adresse_projet"]
        date_candidature = infos_tableau["date_candidature"]

        # Si le numéro de projet n'est pas dans le tableau,
        # on prend l'identifiant depuis l'URL.
        if not numero_projet:
            numero_projet = str(projet_id_url)

        latitude, longitude = extraire_coordonnees(soup, html)

        projets_data.append(
            [
                numero_projet,
                lien,
                nom_projet,
                categorie_concours,
                categorie,
                adresse_precise,
                code_postal,
                commune,
                departement,
                region,
                date_candidature,
                latitude,
                longitude,
            ]
        )

    except Exception as erreur:
        # Les erreurs sont collectees puis resumees a la fin de la cellule.
        projets_erreurs.append([projet_id_url, lien, str(erreur)])


# 2. Sauvegarde projets.csv


colonnes_projets = [
    "projet_id",
    "url_projet",
    "nom_projet",
    "categorie_concours",
    "categorie",
    "adresse_precise",
    "code_postal",
    "commune",
    "departement",
    "region",
    "date_candidature",
    "latitude",
    "longitude",
]

with open("projets.csv", "w", newline="", encoding="utf-8-sig") as f_output:
    csv_output = csv.writer(f_output)
    csv_output.writerow(colonnes_projets)
    csv_output.writerows(projets_data)


# 2.1 Correction commune / departement / region a partir du code postal et du nom
# La resolution administrative (session avec nouvelles tentatives, cache, levee des
# homonymes) est definie une seule fois dans la cellule des imports et partagee avec 2.2.


def corriger_localisations_projets(fichier_csv="projets.csv", nombre_tentatives=3):
    df_localisations = pd.read_csv(fichier_csv, encoding="utf-8-sig", dtype=str)

    # Les valeurs de la page sont conservees : la correction repart toujours
    # des colonnes _original, la cellule peut donc etre relancee sans cumuler.
    for colonne in ["code_postal", "commune", "departement", "region"]:
        if colonne not in df_localisations.columns:
            df_localisations[colonne] = ""
        if f"{colonne}_original" not in df_localisations.columns:
            df_localisations[f"{colonne}_original"] = df_localisations[colonne]

    df_localisations["source_correction_localisation"] = ""
    lignes_a_traiter = list(df_localisations.index)

    for tentative in range(1, nombre_tentatives + 1):
        lignes_en_echec = []

        for index in lignes_a_traiter:
            code_postal = normaliser_code_postal(df_localisations.at[index, "code_postal_original"])
            commune_source = nettoyer_texte(df_localisations.at[index, "commune_original"])

            commune_info, code_postal, source = resoudre_localisation_administrative(
                code_postal_source=code_postal, commune_source=commune_source
            )

            df_localisations.at[index, "code_postal"] = code_postal
            df_localisations.at[index, "source_correction_localisation"] = source

            if commune_info is not None:
                df_localisations.at[index, "commune"] = commune_info.get("nom", "")
                df_localisations.at[index, "departement"] = commune_info.get("departement", {}).get("nom", "")
                df_localisations.at[index, "region"] = commune_info.get("region", {}).get("nom", "")
            elif source == "erreur_reseau":
                # Un echec reseau n'est pas mis en cache : la ligne est rejouee.
                lignes_en_echec.append(index)

        if not lignes_en_echec or tentative == nombre_tentatives:
            break

        print(f"{len(lignes_en_echec)} localisations rejouees (tentative {tentative + 1}/{nombre_tentatives})")
        time.sleep(2 * tentative)
        lignes_a_traiter = lignes_en_echec

    df_localisations.to_csv(fichier_csv, index=False, encoding="utf-8-sig")

    resume = df_localisations["source_correction_localisation"].value_counts()
    print("Correction des localisations :", len(df_localisations), "projets")
    print(resume.to_string())
    resumer_erreurs_api("communes")


corriger_localisations_projets("projets.csv")


# 3. Sauvegarde erreurs.csv


if projets_erreurs:
    with open("projets_erreurs.csv", "w", newline="", encoding="utf-8-sig") as f_error:
        csv_error = csv.writer(f_error)
        csv_error.writerow(["projet_id_url", "url_projet", "erreur"])
        csv_error.writerows(projets_erreurs)


# 4. Vérification


print("Nombre de projets extraits :", len(projets_data))
print("Nombre d'erreurs de lecture :", len(projets_erreurs))

for projet_id_url, lien, erreur in projets_erreurs[:5]:
    print("  ", lien, "|", erreur[:120])
if len(projets_erreurs) > 5:
    print("  ... voir projets_erreurs.csv")

if projets_data:
    premier = dict(zip(colonnes_projets, projets_data[0]))
    print("Premier projet :", premier["projet_id"], "|", premier["nom_projet"],
          "|", premier["code_postal"], premier["commune"])


Correction des localisations : 921 projets
source_correction_localisation
postal_et_commune_confirmes              819
commune_confirmee_code_postal_corrige     36
postal_partage_non_resolu                 32
commune_corrigee_par_postal_unique        25
commune_homonyme_non_resolue               8
code_postal_introuvable                    1
Appels API communes: aucune erreur reseau (472 reponses en cache)
Nombre de projets extraits : 921
Nombre d'erreurs de lecture : 0
Premier projet : 1000 | Chalet bois en station | 74920 Combloux


## 2.2 Validation et géocodage fiable des projets

Cette étape applique un contrôle générique, sans règle liée à un projet particulier :

1. conserver les valeurs source (adresse, code postal, commune et coordonnées) ;
2. résoudre la commune officielle et vérifier la compatibilité du code postal, en levant les homonymies de communes à l'aide du code postal puis du département de la page ;
3. construire une adresse de recherche sans suffixe localité contradictoire ;
4. géocoder l'adresse avec le service Géoplateforme actuel ;
5. comparer les coordonnées du site à cette référence avec un seuil adapté au type de résultat ;
6. conserver, corriger ou envoyer la ligne en contrôle manuel avec une raison explicite.

### Hypothèses clés

- Une correspondance par numéro de voie est plus précise qu'une rue, elle-même plus précise qu'un centre communal.
- Un résultat d'adresse n'est accepté que si **la commune et le code postal** correspondent aux valeurs officielles disponibles.
- Une ligne sans référence d'adresse précise n'est jamais considérée comme pleinement validée : elle apparaît dans `projets_localisations_a_verifier.csv`.

### Mise en oeuvre

Le traitement est identique pour chaque projet, il est donc factorise :

- une localisation identique (adresse, code postal, commune, coordonnees source) n'est traitee qu'une seule fois ;
- chaque requete HTTP est mise en cache, et deux requetes d'adresse au maximum sont envoyees par localisation, la recherche s'arretant au premier resultat conforme ;
- les localisations distinctes sont traitees en parallele (8 requetes simultanees) ;
- un echec reseau n'est jamais confondu avec une absence de resultat : l'appel n'est pas mis en cache, la localisation est rejouee, et si l'echec persiste elle est marquee `erreur_reseau` au lieu d'etre declaree introuvable.


In [5]:
# =============================================================================
# 2.2 Validation et geocodage des projets (version simplifiee et parallelisee)
# =============================================================================
# 1. Parametres

FICHIER_PROJETS = "projets.csv"
FICHIER_GEOCODES = "projets_geocodes_complet.csv"
FICHIER_A_VERIFIER = "projets_localisations_a_verifier.csv"
FICHIER_CORRIGEES = "projets_localisations_corrigees.csv"

URL_API_GEOCODAGE = "https://data.geopf.fr/geocodage/search"

SCORE_MINIMUM = 0.45           # score minimal accepte pour un resultat d'adresse
SEUIL_CENTRE_COMMUNE_KM = 35   # ecart tolere entre coordonnees source et centre communal
NB_REQUETES_PARALLELES = 8     # requetes HTTP simultanees

col_adresse = "adresse_precise"
col_code_postal = "code_postal"
col_commune = "commune"
col_lat = "latitude"
col_lon = "longitude"


# 2. Lecture des donnees et conservation des valeurs source

df = pd.read_csv(FICHIER_PROJETS, encoding="utf-8-sig", dtype=str)

for colonne in [col_lat, col_lon]:
    if colonne not in df.columns:
        df[colonne] = pd.NA

colonnes_source = {
    "adresse_precise_source": col_adresse,
    "code_postal_source": (
        "code_postal_original" if "code_postal_original" in df.columns else col_code_postal
    ),
    "commune_source": (
        "commune_original" if "commune_original" in df.columns else col_commune
    ),
    "latitude_source": col_lat,
    "longitude_source": col_lon,
}

for colonne_cible, colonne_origine in colonnes_source.items():
    if colonne_cible not in df.columns:
        df[colonne_cible] = df[colonne_origine]

for colonne in ["latitude_source", "longitude_source"]:
    df[colonne] = pd.to_numeric(df[colonne], errors="coerce")


# 4. Fonctions utilitaires

def premiere_valeur(valeur):
    if isinstance(valeur, list):
        return valeur[0] if valeur else ""
    return valeur


def coordonnees_valides(latitude, longitude):
    try:
        latitude, longitude = float(latitude), float(longitude)
    except (TypeError, ValueError):
        return False
    return -90 <= latitude <= 90 and -180 <= longitude <= 180


def distance_haversine_km(latitude_a, longitude_a, latitude_b, longitude_b):
    lat_a, lon_a, lat_b, lon_b = np.radians(
        [float(latitude_a), float(longitude_a), float(latitude_b), float(longitude_b)]
    )
    terme = (
        np.sin((lat_b - lat_a) / 2) ** 2
        + np.cos(lat_a) * np.cos(lat_b) * np.sin((lon_b - lon_a) / 2) ** 2
    )
    return float(2 * 6371.0088 * np.arcsin(np.sqrt(terme)))


def seuil_validation_km(type_resultat):
    """Seuil de coherence adapte a la precision du resultat de geocodage."""

    type_resultat = normaliser_chaine(type_resultat)
    if type_resultat in {"housenumber", "numero"}:
        return 1.5
    if type_resultat in {"street", "rue", "locality", "lieu dit"}:
        return 10.0
    return 20.0


def centre_commune(commune_info):
    coordonnees = (commune_info or {}).get("centre", {}).get("coordinates", [])
    return (float(coordonnees[1]), float(coordonnees[0])) if len(coordonnees) >= 2 else None


# 5. Geocodage de l'adresse : 2 requetes au maximum, arret au premier resultat valide

def construire_requetes_adresse(adresse, code_postal="", commune=""):
    adresse = nettoyer_texte(adresse)
    code_postal = normaliser_code_postal(code_postal)
    commune = nettoyer_texte(commune)

    # Le suffixe situe apres le premier code postal est supprime : il contient
    # parfois une commune ou un code postal contradictoire issu de la page source.
    adresse_rue = re.split(r"\b\d{5}\b", adresse, maxsplit=1)[0].strip(" ,;-")
    if est_vide(adresse_rue):
        adresse_rue = adresse

    requetes = []
    if commune:
        if code_postal:
            requetes.append(f"{adresse_rue}, {code_postal} {commune}, France")
        requetes.append(f"{adresse_rue}, {commune}, France")
    else:
        requetes.append(f"{adresse_rue}, France")
    return list(dict.fromkeys(requetes))


def resultat_adresse_valide(properties, code_postal="", commune="", commune_info=None):
    """La commune ET le code postal renvoyes doivent rester compatibles avec l'officiel."""

    if float(properties.get("score", 0) or 0) < SCORE_MINIMUM:
        return False

    codes_acceptes = set(codes_postaux_officiels(commune_info))
    if normaliser_code_postal(code_postal):
        codes_acceptes.add(normaliser_code_postal(code_postal))
    noms_acceptes = [nom for nom in (commune, (commune_info or {}).get("nom", "")) if nom]

    code_retour = normaliser_code_postal(premiere_valeur(properties.get("postcode", "")))
    commune_retour = premiere_valeur(properties.get("city", ""))

    code_ok = not codes_acceptes or code_retour in codes_acceptes
    commune_ok = not noms_acceptes or any(
        communes_equivalentes(commune_retour, nom) for nom in noms_acceptes
    )
    return code_ok and commune_ok


def geocoder_adresse_projet(adresse, code_postal="", commune="", commune_info=None):
    if est_vide(adresse):
        return None

    for requete in construire_requetes_adresse(adresse, code_postal, commune):
        donnees = appeler_api_json(
            URL_API_GEOCODAGE, {"q": requete, "limit": 5, "index": "address"}
        )

        resultats = []
        for feature in (donnees or {}).get("features", []):
            properties = feature.get("properties", {})
            coordonnees = feature.get("geometry", {}).get("coordinates", [])
            if len(coordonnees) < 2:
                continue
            if not resultat_adresse_valide(properties, code_postal, commune, commune_info):
                continue
            resultats.append(
                {
                    "latitude": float(coordonnees[1]),
                    "longitude": float(coordonnees[0]),
                    "label": nettoyer_texte(properties.get("label", "")),
                    "score": float(properties.get("score", 0) or 0),
                    "type": nettoyer_texte(properties.get("type", "")),
                    "commune": nettoyer_texte(premiere_valeur(properties.get("city", ""))),
                    "code_postal": normaliser_code_postal(
                        premiere_valeur(properties.get("postcode", ""))
                    ),
                    "code_insee": nettoyer_texte(premiere_valeur(properties.get("citycode", ""))),
                }
            )

        # Des qu'une requete fournit un resultat conforme, les suivantes sont inutiles.
        if resultats:
            return max(resultats, key=lambda resultat: resultat["score"])

    return None


# 6. Traitement d'une localisation : conserver, corriger ou signaler

def traiter_localisation(cle):
    adresse_source, code_source, commune_source, latitude_source, longitude_source = cle
    source_disponible = coordonnees_valides(latitude_source, longitude_source)

    resultat = {
        col_code_postal: code_source,
        col_commune: commune_source,
        col_lat: pd.NA,
        col_lon: pd.NA,
        "source_geocodage": "",
        "statut_qa_localisation": "non_resolue",
        "raison_correction_localisation": "aucune_reference_exploitable",
        "coordonnees_corrigees": False,
    }

    def enregistrer_commune(commune_info, code_postal):
        resultat.update(
            {
                col_code_postal: code_postal,
                col_commune: nettoyer_texte(commune_info.get("nom", "")),
                "code_insee": nettoyer_texte(commune_info.get("code", "")),
                "departement": commune_info.get("departement", {}).get("nom", ""),
                "region": commune_info.get("region", {}).get("nom", ""),
            }
        )

    def enregistrer_position(latitude, longitude, origine, statut, raison, corrigee=False):
        resultat.update(
            {
                col_lat: latitude,
                col_lon: longitude,
                "source_geocodage": origine,
                "statut_qa_localisation": statut,
                "raison_correction_localisation": raison,
                "coordonnees_corrigees": corrigee,
            }
        )

    # a. Commune officielle
    commune_info, code_officiel, statut_admin = resoudre_localisation_administrative(
        code_postal_source=code_source, commune_source=commune_source
    )
    if commune_info is not None:
        enregistrer_commune(commune_info, code_officiel)

    # b. Reference d'adresse
    reference = geocoder_adresse_projet(
        adresse=adresse_source,
        code_postal=code_officiel,
        commune=resultat[col_commune],
        commune_info=commune_info,
    )

    if reference is not None:
        resultat.update(
            {
                "adresse_geocodee": reference["label"],
                "score_geocodage": reference["score"],
                "type_resultat_geocodage": reference["type"],
                "commune_geocodee": reference["commune"],
                "code_postal_geocode": reference["code_postal"],
            }
        )

        # Le resultat a passe le controle commune + codes postaux officiels : son code
        # postal est plus precis qu'un code generique de commune (Lyon 69000).
        if reference["code_postal"]:
            resultat[col_code_postal] = reference["code_postal"]
            if commune_info is not None and reference["code_postal"] != code_officiel:
                statut_admin += "|code_postal_precise_par_adresse"

        # Si la source ne permettait pas de resoudre la commune, celle renvoyee par le
        # geocodeur est confirmee une seconde fois par l'API administrative.
        if commune_info is None and reference["commune"]:
            commune_info, code_confirme, statut_confirme = resoudre_localisation_administrative(
                code_postal_source=reference["code_postal"],
                commune_source=reference["commune"],
            )
            if commune_info is not None:
                code_officiel = reference["code_postal"] or code_confirme
                enregistrer_commune(commune_info, code_officiel)
                statut_admin = f"administration_resolue_par_adresse|{statut_confirme}"

    resultat["statut_resolution_administrative"] = statut_admin

    # c. Distances de controle
    centre = centre_commune(commune_info)
    distance_centre = None
    if source_disponible and centre is not None:
        distance_centre = distance_haversine_km(
            latitude_source, longitude_source, centre[0], centre[1]
        )
        resultat["distance_source_centre_km"] = round(distance_centre, 3)

    distance_reference, seuil = None, None
    if source_disponible and reference is not None:
        distance_reference = distance_haversine_km(
            latitude_source, longitude_source, reference["latitude"], reference["longitude"]
        )
        seuil = seuil_validation_km(reference["type"])
        resultat["distance_source_reference_km"] = round(distance_reference, 3)
        resultat["seuil_validation_km"] = seuil

    # d. Decision : conserver la source, la corriger, ou signaler la ligne
    if reference is not None and commune_info is None:
        enregistrer_position(
            reference["latitude"], reference["longitude"],
            "geoplateforme_adresse_non_confirmee",
            "a_verifier_admin_non_resolue",
            "commune_geocodee_non_confirmee_par_api_administrative",
            corrigee=True,
        )

    elif reference is not None and distance_reference is not None and distance_reference <= seuil:
        enregistrer_position(
            latitude_source, longitude_source,
            "source_site_validee_adresse",
            "validee_source_vs_adresse",
            "coordonnees_source_confirmees_par_reference",
        )

    elif reference is not None:
        enregistrer_position(
            reference["latitude"], reference["longitude"],
            "geoplateforme_adresse",
            "corrigee_par_adresse",
            "source_trop_eloignee_de_reference" if source_disponible else "coordonnees_source_absentes",
            corrigee=True,
        )

    # Sans reference d'adresse, les coordonnees source proches de la commune sont
    # conservees (elles ne sont jamais considerees comme validees).
    elif source_disponible and (
        distance_centre is None or distance_centre <= SEUIL_CENTRE_COMMUNE_KM
    ):
        enregistrer_position(
            latitude_source, longitude_source,
            "source_site_non_confirmee",
            "a_verifier_admin_non_resolue" if commune_info is None else "a_verifier_source_sans_reference",
            "adresse_non_geocodee_source_conservee",
        )

    elif centre is not None:
        enregistrer_position(
            centre[0], centre[1],
            "centre_commune",
            "a_verifier_centre_commune",
            "source_hors_commune_reference_adresse_absente"
            if source_disponible
            else "adresse_et_coordonnees_source_absentes",
            corrigee=True,
        )

    return resultat


# 7. Execution : une seule fois par localisation distincte, en parallele

cles_lignes = list(
    zip(
        df["adresse_precise_source"].map(nettoyer_texte),
        df["code_postal_source"].map(normaliser_code_postal),
        df["commune_source"].map(nettoyer_texte),
        df["latitude_source"],
        df["longitude_source"],
    )
)
cles_uniques = list(dict.fromkeys(cles_lignes))

print(f"{len(df)} projets, {len(cles_uniques)} localisations distinctes a traiter...")
debut = time.time()

STATUTS_RESOLUS = {"validee_source_vs_adresse", "corrigee_par_adresse"}
resultats = {}
cles_a_traiter = cles_uniques

for tentative in range(1, 4):
    erreurs_avant = len(erreurs_api)

    with ThreadPoolExecutor(max_workers=NB_REQUETES_PARALLELES) as executeur:
        resultats.update(zip(cles_a_traiter, executeur.map(traiter_localisation, cles_a_traiter)))

    # Les reponses reussies sont en cache : un nouveau passage ne rejoue que les
    # appels tombes en erreur reseau.
    if len(erreurs_api) == erreurs_avant or tentative == 3:
        break

    cles_a_traiter = [
        cle for cle in cles_uniques
        if resultats[cle]["statut_qa_localisation"] not in STATUTS_RESOLUS
    ]
    if not cles_a_traiter:
        break

    print(f"{len(cles_a_traiter)} localisations rejouees (tentative {tentative + 1}/3)")
    time.sleep(2 * tentative)

df_resultats = pd.DataFrame([resultats[cle] for cle in cles_lignes], index=df.index)

for colonne in df_resultats.columns:
    valeurs = df_resultats[colonne]
    # departement et region ne sont ecrases que si la commune officielle a ete resolue
    if colonne in {"departement", "region"} and colonne in df.columns:
        valeurs = valeurs.where(valeurs.notna(), df[colonne])
    df[colonne] = valeurs

print(f"Geocodage termine en {time.time() - debut:.1f} s")
resumer_erreurs_api("geocodage")


# 8. Controles invariants et fichiers d'audit

latitude_finale = pd.to_numeric(df[col_lat], errors="coerce")
longitude_finale = pd.to_numeric(df[col_lon], errors="coerce")

if not (latitude_finale.notna() == longitude_finale.notna()).all():
    raise ValueError("Certaines lignes ne possedent qu'une seule coordonnee finale.")

if (
    latitude_finale.notna()
    & (~latitude_finale.between(-90, 90) | ~longitude_finale.between(-180, 180))
).any():
    raise ValueError("Des coordonnees finales sont hors des plages latitude / longitude.")

masque_a_verifier = ~df["statut_qa_localisation"].isin(STATUTS_RESOLUS)
masque_corrige = (
    df["coordonnees_corrigees"].fillna(False).astype(bool)
    | (
        df[col_code_postal].map(normaliser_code_postal)
        != df["code_postal_source"].map(normaliser_code_postal)
    )
    | pd.Series(
        [
            not (est_vide(finale) and est_vide(origine))
            and not communes_equivalentes(finale, origine)
            for finale, origine in zip(df[col_commune], df["commune_source"])
        ],
        index=df.index,
    )
)

colonnes_audit = [
    "projet_id", "nom_projet", "url_projet",
    "adresse_precise_source", "code_postal_source", "commune_source",
    "latitude_source", "longitude_source",
    col_code_postal, col_commune, "departement", "region", col_lat, col_lon,
    "adresse_geocodee", "score_geocodage", "type_resultat_geocodage",
    "distance_source_reference_km", "seuil_validation_km",
    "statut_resolution_administrative", "statut_qa_localisation",
    "raison_correction_localisation",
]
colonnes_audit = [colonne for colonne in colonnes_audit if colonne in df.columns]

df.to_csv(FICHIER_GEOCODES, index=False, encoding="utf-8-sig")
df.loc[masque_a_verifier, colonnes_audit].to_csv(FICHIER_A_VERIFIER, index=False, encoding="utf-8-sig")
df.loc[masque_corrige, colonnes_audit].to_csv(FICHIER_CORRIGEES, index=False, encoding="utf-8-sig")

print("Pipeline de localisation termine :", len(df), "projets")
print(df["statut_qa_localisation"].value_counts(dropna=False).to_string())
print("Corrections automatiques auditees :", int(masque_corrige.sum()))
print("Lignes a verifier manuellement :", int(masque_a_verifier.sum()))
print("Fichiers crees :", FICHIER_GEOCODES, "/", FICHIER_CORRIGEES, "/", FICHIER_A_VERIFIER)


921 projets, 919 localisations distinctes a traiter...
275 localisations rejouees (tentative 2/3)
275 localisations rejouees (tentative 3/3)
Geocodage termine en 64.0 s
Appels API geocodage: 6 echecs reseau rejoues (HTTPError x6)
Pipeline de localisation termine : 921 projets
statut_qa_localisation
corrigee_par_adresse                379
validee_source_vs_adresse           267
a_verifier_centre_commune           129
a_verifier_source_sans_reference    106
non_resolue                          22
a_verifier_admin_non_resolue         18
Corrections automatiques auditees : 530
Lignes a verifier manuellement : 275
Fichiers crees : projets_geocodes_complet.csv / projets_localisations_corrigees.csv / projets_localisations_a_verifier.csv


### 2.2.1 Tests de non-régression des deux erreurs observées

Ces tests rejouent les deux combinaisons qui avaient échappé à l'ancien traitement. Ils ne participent pas à la correction des données : ils vérifient uniquement que le pipeline générique continue à détecter ces classes d'erreurs.


In [6]:
cas_non_regression = [
    {
        "cas": "code_postal_incompatible_avec_commune_exacte",
        "adresse": "1515 Rte de Lormay 73470 LE GRAND BORNAND",
        "code_postal": "73470",
        "commune": "LE GRAND BORNAND",
        "latitude_source": 45.93553389448995,
        "longitude_source": 7.324515609375015,
        "code_postal_attendu": "74450",
        "commune_attendue": "Le Grand-Bornand",
    },
    {
        "cas": "suffixe_adresse_trompeur_et_coordonnees_hors_reference",
        "adresse": "269, rue des Egratz 74190 LE FAYET ST GERVAIS LES BAINS",
        "code_postal": "74190",
        "commune": "LE FAYET ST GERVAIS LES BAINS",
        "latitude_source": 45.76722443872639,
        "longitude_source": 7.492661894531265,
        "code_postal_attendu": "74190",
        "commune_attendue": "Passy",
    },
]

resultats_non_regression = []

for cas in cas_non_regression:
    commune_info, code_officiel, statut_admin = resoudre_localisation_administrative(
        code_postal_source=cas["code_postal"],
        commune_source=cas["commune"],
    )
    commune_officielle = nettoyer_texte((commune_info or {}).get("nom", ""))
    reference = geocoder_adresse_projet(
        adresse=cas["adresse"],
        code_postal=code_officiel,
        commune=commune_officielle,
        commune_info=commune_info,
    )

    assert commune_info is not None, f"Commune non résolue : {cas['cas']}"
    assert code_officiel == cas["code_postal_attendu"], cas["cas"]
    assert communes_equivalentes(commune_officielle, cas["commune_attendue"]), cas["cas"]
    assert reference is not None, f"Adresse non géocodée : {cas['cas']}"

    distance_km = distance_haversine_km(
        cas["latitude_source"],
        cas["longitude_source"],
        reference["latitude"],
        reference["longitude"],
    )
    seuil_km = seuil_validation_km(reference["type"])
    erreur_detectee = distance_km > seuil_km
    assert erreur_detectee, f"Ancienne coordonnée acceptée à tort : {cas['cas']}"

    resultats_non_regression.append(
        {
            "cas": cas["cas"],
            "statut_administratif": statut_admin,
            "code_postal_resolu": code_officiel,
            "commune_resolue": commune_officielle,
            "distance_ancienne_coord_reference_km": round(distance_km, 1),
            "seuil_km": seuil_km,
            "erreur_detectee": erreur_detectee,
        }
    )

df_tests_non_regression = pd.DataFrame(resultats_non_regression)
print(df_tests_non_regression.to_string(index=False))
print("Tests de non-régression réussis :", len(df_tests_non_regression), "/", len(cas_non_regression))


                                                   cas                  statut_administratif code_postal_resolu  commune_resolue  distance_ancienne_coord_reference_km  seuil_km  erreur_detectee
          code_postal_incompatible_avec_commune_exacte commune_confirmee_code_postal_corrige              74450 Le Grand-Bornand                                  63.7       1.5             True
suffixe_adresse_trompeur_et_coordonnees_hors_reference    commune_corrigee_par_postal_unique              74190            Passy                                  62.0      10.0             True
Tests de non-régression réussis : 2 / 2


## 2.3 Affichage des projets sur la carte

In [7]:
df = pd.read_csv(
    "projets_geocodes_complet.csv",
    encoding="utf-8-sig",
    dtype={"code_postal": "string"},
)

col_lat = "latitude"
col_lon = "longitude"

# only keep line with lat and lon
df_map = df.dropna(subset=[col_lat, col_lon]).copy()

df_map[col_lat] = df_map[col_lat].astype(float)
df_map[col_lon] = df_map[col_lon].astype(float)

# centre de la carte
centre_lat = df_map[col_lat].mean()
centre_lon = df_map[col_lon].mean()

m = folium.Map(location=[centre_lat, centre_lon], zoom_start=9)

marker_cluster = MarkerCluster().add_to(m)

for index, row in df_map.iterrows():

    nom = row["nom_projet"]
    lat = row[col_lat]
    lon = row[col_lon]
    adresse = row["adresse_precise"]
    commune = row["commune"]
    code_postal = row["code_postal"]

    popup_text = f"""
    <b>Nom du projet :</b> {nom}<br>
    <b>Adresse :</b> {adresse}<br>
    <b>Commune :</b> {commune}<br>
    <b>Code postal :</b> {code_postal}
    """

    folium.Marker(
        location=[lat, lon],
        popup=folium.Popup(popup_text, max_width=400),
        tooltip=str(commune),
    ).add_to(marker_cluster)

m


## 3.1 Extraction des acteurs associés aux projets

In [8]:
base_site = "https://www.panoramabois.fr"

acteurs_data = []


df_liens_projets = pd.read_csv("liens_projets.csv", dtype=str)
liens_projets = df_liens_projets["url_projet"].dropna().astype(str).str.strip()
liens_projets = liens_projets[liens_projets != ""].drop_duplicates().tolist()



def extraire_groupe_acteur(texte):
    texte = nettoyer_texte(texte)

    match = re.match(r"(.+é)\s*\((\d+)\)\s*$", texte)

    if match:
        return match.group(1).strip(), match.group(2)

    return texte, ""


def extraire_nom_departement_acteur(texte):

    texte = nettoyer_texte(texte)
    texte = texte.replace("*", "").strip()

    match = re.match(r"(.+é)\s*\(([^()]*)\)\s*$", texte)

    if match:
        return match.group(1).strip(), match.group(2).strip()

    return texte, ""


# 1. Boucle principale


for lien in liens_projets:

    match_id_projet = re.search(r"/projets/(\d+)", lien)
    projet_id_url = match_id_projet.group(1) if match_id_projet else ""

    print("Lecture de :", lien)

    try:
        soup, html = charger_page_projet(lien)

        # Acteurs du projet

        acteurs_content = soup.select_one("div.Project-actors-content")

        if acteurs_content:
            groupe_acteur = ""
            nombre_acteurs_groupe = ""
            role_acteur = ""

            for element in acteurs_content.find_all(["h3", "h4", "p"], recursive=False):

                classes = element.get("class", [])

                # Exemple : Maître d'ouvrage (1)
                if "Project-actors-title2" in classes:
                    groupe_acteur, nombre_acteurs_groupe = extraire_groupe_acteur(
                        element
                    )

                # Exemple : Maître d'ouvrage - État ou collectivité
                elif "Project-actors-title3" in classes:
                    role_acteur = nettoyer_texte(element)

                # Exemple : Conseil Départemental de la Saône et Loire (71)
                elif "Project-actors-text" in classes:
                    liens_acteurs = element.find_all("a")

                    for a in liens_acteurs:
                        acteur_texte = nettoyer_texte(a)
                        acteur_nom, acteur_departement = (
                            extraire_nom_departement_acteur(acteur_texte)
                        )

                        href = a.get("href") or ""
                        acteur_url = ""

                        if href and not href.startswith("javascript"):
                            acteur_url = urljoin(base_site, href)

                        if href.startswith("javascript") or "*" in acteur_texte:
                            entreprise_inactive = "oui"
                        else:
                            entreprise_inactive = "non"

                        acteurs_data.append(
                            [
                                str(projet_id_url),
                                lien,
                                groupe_acteur,
                                nombre_acteurs_groupe,
                                role_acteur,
                                acteur_nom,
                                acteur_departement,
                                acteur_url,
                                entreprise_inactive,
                            ]
                        )

        time.sleep(0.5)

    except Exception as e:
        print("Erreur sur :", lien)
        print(e)


# 2. Sauvegarde acteurs_projets.csv


colonnes_acteurs = [
    "projet_id",
    "url_projet",
    "groupe_acteur",
    "nombre_acteurs_groupe",
    "role_acteur",
    "acteur_nom",
    "acteur_departement",
    "acteur_url",
    "entreprise_inactive",
]

with open("acteurs_projets.csv", "w", newline="", encoding="utf-8-sig") as f_output:
    csv_output = csv.writer(f_output)
    csv_output.writerow(colonnes_acteurs)
    csv_output.writerows(acteurs_data)

# Sauvegarde des liens acteurs pour l'extraction détaillée
if acteurs_data:
    df_acteurs_projets = pd.DataFrame(acteurs_data, columns=colonnes_acteurs)
    df_liens_acteurs = df_acteurs_projets[["acteur_url"]].copy()
    df_liens_acteurs["acteur_url"] = df_liens_acteurs["acteur_url"].astype(str).str.strip()
    df_liens_acteurs = df_liens_acteurs[df_liens_acteurs["acteur_url"] != ""]
    df_liens_acteurs = df_liens_acteurs.drop_duplicates()
else:
    df_liens_acteurs = pd.DataFrame(columns=["acteur_url"])

df_liens_acteurs.to_csv("liens_acteurs.csv", index=False, encoding="utf-8-sig")

print("Nombre d'acteurs extraits :", len(acteurs_data))
print("Liens acteurs enregistrés dans : liens_acteurs.csv")


Lecture de : https://www.panoramabois.fr/projets/1000
Lecture de : https://www.panoramabois.fr/projets/1002
Lecture de : https://www.panoramabois.fr/projets/1014
Lecture de : https://www.panoramabois.fr/projets/1021
Lecture de : https://www.panoramabois.fr/projets/1024
Lecture de : https://www.panoramabois.fr/projets/1025
Lecture de : https://www.panoramabois.fr/projets/1049
Lecture de : https://www.panoramabois.fr/projets/1050
Lecture de : https://www.panoramabois.fr/projets/1052
Lecture de : https://www.panoramabois.fr/projets/1062
Lecture de : https://www.panoramabois.fr/projets/1066
Lecture de : https://www.panoramabois.fr/projets/1067
Lecture de : https://www.panoramabois.fr/projets/1086
Lecture de : https://www.panoramabois.fr/projets/1137
Lecture de : https://www.panoramabois.fr/projets/1220
Lecture de : https://www.panoramabois.fr/projets/1238
Lecture de : https://www.panoramabois.fr/projets/1239
Lecture de : https://www.panoramabois.fr/projets/1252
Lecture de : https://www.pan

## 3.2 Informations sur les acteurs

In [ ]:
# 0. Paramètres généraux


base_site = "https://www.panoramabois.fr"

input_csv = "liens_acteurs.csv"  # Fichier des liens acteurs
output_acteurs_csv = "acteurs_infos_completes.csv"
output_portfolio_csv = "portfolio_acteurs.csv"

# Nom de la colonne contenant les liens des acteurs
# Exemples : url_acteur, lien_acteur, acteur_lien ou lien
actor_url_column = "acteur_url"

headers = {"User-Agent": "Mozilla/5.0"}


# 2. Extraction du code postal, de la commune, du département et de la région


def nettoyer_suffixe_cedex(commune):
    commune = nettoyer_texte(commune)
    commune = re.sub(r"\s+CEDEX(?:\s+\d+)?\s*$", "", commune, flags=re.IGNORECASE)
    return re.sub(r"\s+", " ", commune).strip()


def extraire_code_postal_commune_dep_region(texte):
    code_postal = ""
    commune = ""
    departement = ""
    region = ""

    texte = texte.replace("\xa0", " ")
    texte = re.sub(r"\s+", " ", texte).strip()

    # Exemple :
    # 38081 L'ISLE D'ABEAU CEDEX - Is?re - Auvergne-Rh?ne-Alpes
    parties = [p.strip() for p in re.split(r"\s+-\s+", texte) if p.strip()]

    if len(parties) >= 1:
        match = re.match(r"^(\d{5})\s+(.+)$", parties[0])
        if match:
            code_postal = match.group(1).strip()
            commune = nettoyer_suffixe_cedex(match.group(2))
        else:
            commune = nettoyer_suffixe_cedex(parties[0])

    if len(parties) >= 2:
        departement = parties[1]

    if len(parties) >= 3:
        region = parties[2]

    return code_postal, commune, departement, region


def extraire_coordonnees_acteur_source(html):
    if est_vide(html):
        return "", ""

    match = re.search(
        r"new\s+google\.maps\.LatLng\s*\(\s*([+-]?\d+(?:\.\d+)?)\s*,\s*([+-]?\d+(?:\.\d+)?)\s*\)",
        html,
        flags=re.IGNORECASE,
    )

    if not match:
        return "", ""

    return match.group(1), match.group(2)


# 3. Extraction des informations de la page d'un acteur


def extraire_page_acteur(html, url_acteur=""):
    soup = BeautifulSoup(html, "html.parser")

    acteur = {
        "url_acteur": url_acteur,
        "id_acteur": "",
        "nom_acteur": "",
        "code_postal": "",
        "commune": "",
        "departement": "",
        "region": "",
        "adresse_detaillee": "",
        "latitude": "",
        "longitude": "",
        "source_coordonnees": "",
        "telephone": "",
        "site_internet": "",
        "interprofession_referente": "",
        "dernier_projet_titre": "",
        "dernier_projet_id": "",
        "dernier_projet_lien": "",
        "statut_scraping": "ok",
    }

    portfolio = []

    # id acteur depuis URL
    match_id_acteur = re.search(r"/acteurs/(\d+)", url_acteur)
    if match_id_acteur:
        acteur["id_acteur"] = match_id_acteur.group(1)

    # Nom acteur

    titre = soup.select_one("h1.Project-title")
    acteur["nom_acteur"] = nettoyer_texte(titre)

    # Adresse courte : CP / commune / département / région

    adresse_resume = soup.select_one("div.Project-address")
    texte_adresse_resume = nettoyer_texte(adresse_resume)

    (
        acteur["code_postal"],
        acteur["commune"],
        acteur["departement"],
        acteur["region"],
    ) = extraire_code_postal_commune_dep_region(texte_adresse_resume)

    latitude_source, longitude_source = extraire_coordonnees_acteur_source(html)
    if latitude_source and longitude_source:
        acteur["latitude"] = latitude_source
        acteur["longitude"] = longitude_source
        acteur["source_coordonnees"] = "page_source"

    # Bloc contact : adresse détaillée + téléphone

    grid = soup.select_one("div.Grid")

    if grid:
        premiere_colonne = grid.find("div", recursive=False)

        if premiere_colonne:
            premier_p = premiere_colonne.find("p")

            if premier_p:
                lignes = [
                    ligne.strip()
                    for ligne in premier_p.get_text("\n", strip=True).split("\n")
                    if ligne.strip()
                ]

                telephone_regex = r"(?:\+33|0)\s*\d(?:[\s.-]*\d{2}){4}"
                lignes_adresse = []

                for ligne in lignes:
                    ligne = nettoyer_texte(ligne)

                    if re.fullmatch(telephone_regex, ligne):
                        acteur["telephone"] = ligne
                    else:
                        lignes_adresse.append(ligne)

                acteur["adresse_detaillee"] = " ".join(lignes_adresse)

            # Site internet

            for p in premiere_colonne.find_all("p"):
                texte_p = nettoyer_texte(p)

                if "Site internet" in texte_p:
                    lien_site = p.select_one("a[href]")
                    if lien_site:
                        acteur["site_internet"] = lien_site.get("href", "").strip()

                if "Interprofession référente" in texte_p:
                    acteur["interprofession_referente"] = re.sub(
                        r"^Interprofession référente\s*:\s*", "", texte_p
                    ).strip()

    # Dernier projet

    dernier_card = soup.select_one("div.ProjectCard")

    if dernier_card:
        titre_projet = dernier_card.select_one(".ProjectCard-title")
        acteur["dernier_projet_titre"] = nettoyer_texte(titre_projet)

        href = ""

        lien_projet = dernier_card.select_one("a[href*='/projets/']")
        if lien_projet:
            href = lien_projet.get("href", "")
        else:
            onclick = dernier_card.get("onclick", "")
            match_onclick = re.search(r"window\.location=['\"]([^'\"]+)['\"]", onclick)
            if match_onclick:
                href = match_onclick.group(1)

        if href:
            acteur["dernier_projet_lien"] = urljoin(base_site, href)

            match_id = re.search(r"/projets/(\d+)", href)
            if match_id:
                acteur["dernier_projet_id"] = match_id.group(1)

    # Portfolio

    for lien in soup.select(".ProjectSlider a[href*='/projets/']"):
        href = lien.get("href", "")
        titre = nettoyer_texte(lien)

        match_id = re.search(r"/projets/(\d+)", href)

        portfolio.append(
            {
                "id_acteur": acteur["id_acteur"],
                "nom_acteur": acteur["nom_acteur"],
                "url_acteur": url_acteur,
                "projet_id": match_id.group(1) if match_id else "",
                "projet_titre": titre,
                "projet_lien": urljoin(base_site, href),
            }
        )

    return acteur, portfolio


# 4. Lecture du CSV et traitement de chaque page d'acteur


df = pd.read_csv(input_csv)

print("Colonnes disponibles dans le fichier CSV :")
print(df.columns.tolist())

if actor_url_column not in df.columns:
    raise ValueError(
        f"Colonne introuvable : {actor_url_column}. Vérifiez le nom de la colonne dans le CSV."
    )

# Supprimer les liens vides et les doublons
urls_acteurs = df[actor_url_column].dropna().astype(str).str.strip()

urls_acteurs = urls_acteurs[urls_acteurs != ""].unique().tolist()

# Convertir les liens relatifs, par exemple /acteurs/11562, en liens absolus
urls_acteurs = [urljoin(base_site, url) for url in urls_acteurs]

print(f"{len(urls_acteurs)} liens d’acteurs trouvés")


acteurs_data = []
portfolio_data = []


for i, url in enumerate(urls_acteurs, start=1):
    print(f"[{i}/{len(urls_acteurs)}] Lecture : {url}")

    try:
        reponse = requests.get(url, headers=headers, timeout=20)
        reponse.raise_for_status()

        html = reponse.text

        acteur, portfolio = extraire_page_acteur(html, url_acteur=url)

        acteurs_data.append(acteur)
        portfolio_data.extend(portfolio)

        time.sleep(1)

    except Exception as e:
        print("Erreur :", url, e)

        acteurs_data.append(
            {
                "url_acteur": url,
                "id_acteur": "",
                "nom_acteur": "",
                "code_postal": "",
                "commune": "",
                "departement": "",
                "region": "",
                "adresse_detaillee": "",
                "latitude": "",
                "longitude": "",
                "source_coordonnees": "",
                "telephone": "",
                "site_internet": "",
                "interprofession_referente": "",
                "dernier_projet_titre": "",
                "dernier_projet_id": "",
                "dernier_projet_lien": "",
                "statut_scraping": f"erreur : {e}",
            }
        )


# 5. Enregistrement des résultats


df_acteurs = pd.DataFrame(acteurs_data)
df_portfolio = pd.DataFrame(portfolio_data)

df_acteurs.to_csv(output_acteurs_csv, index=False, encoding="utf-8-sig")
df_portfolio.to_csv(output_portfolio_csv, index=False, encoding="utf-8-sig")

print("Traitement terminé.")
print(f"Informations sur les acteurs enregistrées dans : {output_acteurs_csv}")
print(f"Portefeuille des acteurs enregistré dans : {output_portfolio_csv}")


Colonnes disponibles dans le fichier CSV :
['acteur_url']
1968 liens d’acteurs trouvés
[1/1968] Lecture : https://www.panoramabois.fr/projets/acteurs/1943
[2/1968] Lecture : https://www.panoramabois.fr/projets/acteurs/3416
[3/1968] Lecture : https://www.panoramabois.fr/projets/acteurs/164
[4/1968] Lecture : https://www.panoramabois.fr/projets/acteurs/3417
[5/1968] Lecture : https://www.panoramabois.fr/projets/acteurs/2537
[6/1968] Lecture : https://www.panoramabois.fr/projets/acteurs/3462
[7/1968] Lecture : https://www.panoramabois.fr/projets/acteurs/96
[8/1968] Lecture : https://www.panoramabois.fr/projets/acteurs/2048
[9/1968] Lecture : https://www.panoramabois.fr/projets/acteurs/3463
[10/1968] Lecture : https://www.panoramabois.fr/projets/acteurs/120
[11/1968] Lecture : https://www.panoramabois.fr/projets/acteurs/3464
[12/1968] Lecture : https://www.panoramabois.fr/projets/acteurs/1669
[13/1968] Lecture : https://www.panoramabois.fr/projets/acteurs/4292
[14/1968] Lecture : https://w

## 3.3 Géocodage des acteurs

In [10]:
# 1. Paramétres

fichier_entree = "acteurs_infos_completes.csv"
fichier_sortie = "acteurs_geocodes_complet.csv"


# 2. Lire le fichier CSV

df = pd.read_csv(fichier_entree, encoding="utf-8-sig", dtype={"code_postal": "string"})

print("Colonnes disponibles :")
print(df.columns.tolist())
print("Nombre total d'acteurs :", len(df))


# 3. Définir les colonnes

colonnes_adresse_possibles = ["adresse_precise", "adresse_detaillee"]
col_adresse = next((col for col in colonnes_adresse_possibles if col in df.columns), None)

if col_adresse is None:
    raise ValueError("Aucune colonne d'adresse précise trouvée.")

col_nom = "nom_acteur"
col_code_postal = "code_postal"

if "commune" not in df.columns and "ville" in df.columns:
    df = df.rename(columns={"ville": "commune"})
elif "commune" in df.columns and "ville" in df.columns:
    masque_commune_vide = df["commune"].isna() | (df["commune"].astype(str).str.strip() == "")
    df.loc[masque_commune_vide, "commune"] = df.loc[masque_commune_vide, "ville"]
    df = df.drop(columns=["ville"])

col_commune = "commune"
if col_commune not in df.columns:
    df[col_commune] = ""

col_lat = "latitude"
col_lon = "longitude"

print("Colonne d'adresse utilisée :", col_adresse)
print("Colonne de commune utilisée :", col_commune)


# 4. Préparer les colonnes utiles

for col in [col_lat, col_lon]:
    if col not in df.columns:
        df[col] = pd.NA

for col in ["adresse_geocodee", "source_geocodage", "source_coordonnees", "commune_corrigee"]:
    if col not in df.columns:
        df[col] = ""
    df[col] = df[col].astype("object")

colonnes_a_supprimer = [
    "score_geocodage",
    "code_postal_geocode",
    "type_geocodage",
    "remarque_geocodage",
]

df = df.drop(columns=colonnes_a_supprimer, errors="ignore")

df[col_lat] = pd.to_numeric(df[col_lat], errors="coerce")
df[col_lon] = pd.to_numeric(df[col_lon], errors="coerce")


# 5. Créer une session requests avec retry

session = requests.Session()

strategie_retry = Retry(
    total=4,
    connect=4,
    read=4,
    status=4,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"],
)

adaptateur = HTTPAdapter(max_retries=strategie_retry)
session.mount("https://", adaptateur)
session.mount("http://", adaptateur)
session.headers.update(
    {"User-Agent": "Mozilla/5.0 geocodage-acteurs-panorama-bois/1.0"}
)


# 6. Fonctions communes

import unicodedata


def normaliser_chaine(valeur):
    texte = nettoyer_texte(valeur).casefold()
    texte = unicodedata.normalize("NFD", texte)
    texte = "".join(car for car in texte if unicodedata.category(car) != "Mn")
    texte = re.sub(r"[^a-z0-9]+", " ", texte)
    return re.sub(r"\s+", " ", texte).strip()


def normaliser_nom_commune(valeur):
    """Développer uniquement les abréviations ST et STE comme mots entiers."""
    mots = normaliser_chaine(valeur).split()
    abreviations = {"st": "saint", "ste": "sainte"}
    return " ".join(abreviations.get(mot, mot) for mot in mots)


def communes_equivalentes(commune_a, commune_b):
    nom_a = normaliser_nom_commune(commune_a)
    nom_b = normaliser_nom_commune(commune_b)
    return bool(nom_a and nom_b and nom_a == nom_b)


def choisir_commune(data, commune=""):
    if not data:
        return None

    if not est_vide(commune):
        for commune_info in data:
            if communes_equivalentes(commune_info.get("nom", ""), commune):
                return commune_info

    return data[0]


# 7. Recherche commune officielle


def interroger_api_communes(params):
    url = "https://geo.api.gouv.fr/communes"
    reponse = session.get(url, params=params, timeout=20)
    reponse.raise_for_status()
    return reponse.json()


def trouver_infos_commune(code_postal="", commune=""):
    code_postal = normaliser_code_postal(code_postal)
    commune = nettoyer_texte(commune)

    if est_vide(code_postal) and est_vide(commune):
        return None

    if not est_vide(code_postal):
        params = {
            "codePostal": code_postal,
            "fields": "nom,centre,codesPostaux",
            "format": "json",
        }

        try:
            data_code_postal = interroger_api_communes(params)
        except Exception as e:
            print("  Erreur recherche commune :", code_postal, commune, "|", e)
            data_code_postal = []

        commune_info = choisir_commune(data_code_postal, commune)

        if commune_info is not None:
            codes_postaux = [
                normaliser_code_postal(cp)
                for cp in commune_info.get("codesPostaux", [])
            ]

            if code_postal in codes_postaux:
                return commune_info

    if not est_vide(commune):
        params = {
            "nom": commune,
            "fields": "nom,centre,codesPostaux",
            "format": "json",
        }

        try:
            data_commune = interroger_api_communes(params)
        except Exception as e:
            print("  Erreur recherche commune :", code_postal, commune, "|", e)
            return None

        commune_info = choisir_commune(data_commune, commune)

        if commune_info is not None:
            codes_postaux = [
                normaliser_code_postal(cp)
                for cp in commune_info.get("codesPostaux", [])
            ]

            if est_vide(code_postal) or code_postal in codes_postaux:
                return commune_info

            if communes_equivalentes(commune_info.get("nom", ""), commune):
                return commune_info

    return None


def completer_commune_si_absente(code_postal="", commune=""):
    if not est_vide(commune) or est_vide(code_postal):
        return nettoyer_texte(commune), None

    commune_info = trouver_infos_commune(code_postal=code_postal)

    if commune_info is None:
        return "", None

    return nettoyer_texte(commune_info.get("nom", "")), commune_info


# 8. Géocodage avec adresse précise


def construire_requetes_adresse(adresse, code_postal="", commune="", commune_info=None):
    adresse = nettoyer_texte(adresse)
    code_postal = normaliser_code_postal(code_postal)
    commune = nettoyer_texte(commune)

    codes_postaux = []

    if code_postal:
        codes_postaux.append(code_postal)

    if commune_info:
        for code in commune_info.get("codesPostaux", []) or []:
            code = normaliser_code_postal(code)
            if code and code not in codes_postaux:
                codes_postaux.append(code)

    requetes = []

    for code in codes_postaux:
        adresse_corrigee = adresse

        if code_postal and code != code_postal:
            adresse_corrigee = re.sub(rf"\b{re.escape(code_postal)}\b", code, adresse)

        requetes.append(f"{adresse_corrigee}, {code} {commune}, France")

    if commune:
        requetes.append(f"{adresse}, {commune}, France")

    if not requetes:
        requetes.append(f"{adresse}, France")

    return list(dict.fromkeys(requetes))


def resultat_adresse_valide(properties, code_postal="", commune="", commune_info=None):
    score = properties.get("score", 0) or 0
    commune_retour = properties.get("city", "")
    code_retour = normaliser_code_postal(properties.get("postcode", ""))

    codes_acceptes = set()

    code_postal = normaliser_code_postal(code_postal)
    if code_postal:
        codes_acceptes.add(code_postal)

    noms_acceptes = set()
    if not est_vide(commune):
        noms_acceptes.add(commune)

    if commune_info:
        noms_acceptes.add(commune_info.get("nom", ""))
        codes_acceptes.update(commune_info.get("codesPostaux", []) or [])

    codes_acceptes = {normaliser_code_postal(code) for code in codes_acceptes if code}

    commune_ok = any(
        communes_equivalentes(commune_retour, nom)
        for nom in noms_acceptes
        if not est_vide(nom)
    )
    code_ok = bool(code_retour and code_retour in codes_acceptes)

    return score >= 0.45 and (commune_ok or code_ok)


def geocoder_adresse_acteur(adresse, code_postal="", commune="", commune_info=None):
    if est_vide(adresse):
        return None, None, ""

    url = "https://api-adresse.data.gouv.fr/search/"

    for adresse_complete in construire_requetes_adresse(
        adresse=adresse,
        code_postal=code_postal,
        commune=commune,
        commune_info=commune_info,
    ):
        params = {"q": adresse_complete, "limit": 5}

        features = []

        for tentative in range(3):
            try:
                reponse = session.get(url, params=params, timeout=15)
                reponse.raise_for_status()
                features = reponse.json().get("features", [])
                break

            except Exception as e:
                if tentative == 2:
                    print("  Erreur géocodage adresse :", adresse_complete, "|", e)
                else:
                    time.sleep(0.5)

        for feature in features:
            properties = feature.get("properties", {})

            if not resultat_adresse_valide(
                properties,
                code_postal=code_postal,
                commune=commune,
                commune_info=commune_info,
            ):
                continue

            coords = feature.get("geometry", {}).get("coordinates", [])

            if len(coords) < 2:
                continue

            lon = coords[0]
            lat = coords[1]
            label = properties.get("label", "")

            return lat, lon, label

    return None, None, ""


# 9. Géocodage par centre de commune


def geocoder_centre_commune(code_postal="", commune="", commune_info=None):
    if commune_info is None:
        commune_info = trouver_infos_commune(code_postal=code_postal, commune=commune)

    if commune_info is None:
        return None, None, ""

    centre = commune_info.get("centre", {})
    coords = centre.get("coordinates", [])

    if len(coords) < 2:
        return None, None, commune_info.get("nom", "")

    lon = coords[0]
    lat = coords[1]
    nom_commune = commune_info.get("nom", "")

    return lat, lon, nom_commune


# 10. Boucle principale

nombre_total = len(df)
nombre_deja_geocodes = 0
nombre_geocodes_adresse = 0
nombre_geocodes_commune = 0
nombre_echecs = 0

for compteur, (index, row) in enumerate(df.iterrows(), start=1):
    nom_acteur = nettoyer_texte(row.get(col_nom, ""))
    adresse = nettoyer_texte(row.get(col_adresse, ""))
    code_postal = normaliser_code_postal(row.get(col_code_postal, ""))
    commune = nettoyer_texte(row.get(col_commune, ""))
    latitude_existante = row.get(col_lat)
    longitude_existante = row.get(col_lon)

    print(f"[{compteur}/{nombre_total}] {nom_acteur or 'Acteur sans nom'}")

    if not est_vide(latitude_existante) and not est_vide(longitude_existante):
        source_existante = nettoyer_texte(row.get("source_coordonnees", ""))
        if est_vide(source_existante):
            source_existante = nettoyer_texte(row.get("source_geocodage", "")) or "page_source"
            df.at[index, "source_coordonnees"] = source_existante
        if est_vide(df.at[index, "source_geocodage"]):
            df.at[index, "source_geocodage"] = source_existante
        print("  Coordonn?es d?j? pr?sentes :", source_existante)
        nombre_deja_geocodes += 1
        continue

    commune, commune_info = completer_commune_si_absente(
        code_postal=code_postal,
        commune=commune,
    )

    if not est_vide(commune):
        df.at[index, col_commune] = commune

    latitude = None
    longitude = None
    adresse_geocodee = ""
    source_geocodage = ""

    if not est_vide(adresse):
        print("  Géocodage adresse précise :", adresse)
        latitude, longitude, adresse_geocodee = geocoder_adresse_acteur(
            adresse=adresse,
            code_postal=code_postal,
            commune=commune,
            commune_info=commune_info,
        )

        if latitude is not None and longitude is not None:
            source_geocodage = "geocodage_adresse_detaillee"
            nombre_geocodes_adresse += 1

        time.sleep(0.2)

    if latitude is None or longitude is None:
        if not est_vide(code_postal) or not est_vide(commune):
            print("  Géocodage centre commune :", code_postal, commune)
            latitude, longitude, adresse_geocodee = geocoder_centre_commune(
                code_postal=code_postal,
                commune=commune,
                commune_info=commune_info,
            )

            if latitude is not None and longitude is not None:
                source_geocodage = "centre_commune"
                nombre_geocodes_commune += 1

            time.sleep(0.2)

    if latitude is not None and longitude is not None:
        df.at[index, col_lat] = latitude
        df.at[index, col_lon] = longitude
        df.at[index, "adresse_geocodee"] = adresse_geocodee
        df.at[index, "source_geocodage"] = source_geocodage
        df.at[index, "source_coordonnees"] = source_geocodage

        if source_geocodage == "centre_commune" and est_vide(df.at[index, "commune_corrigee"]):
            df.at[index, "commune_corrigee"] = adresse_geocodee

        print("  Coordonnées trouvées :", latitude, longitude)
    else:
        df.at[index, "adresse_geocodee"] = ""
        df.at[index, "source_geocodage"] = ""
        df.at[index, "source_coordonnees"] = ""
        nombre_echecs += 1
        print("  Coordonnées non trouvées")

    if compteur % 100 == 0:
        df.to_csv(fichier_sortie, index=False, encoding="utf-8-sig")
        print("  Sauvegarde intermédiaire effectuée")


# 11. Sauvegarder tous les acteurs

df.to_csv(fichier_sortie, index=False, encoding="utf-8-sig")


# 12. Résumé

df_sans_coordonnees = df[df[col_lat].isna() | df[col_lon].isna()].copy()

print("\nGéocodage terminé.")
print("Nombre total d'acteurs :", nombre_total)
print("Acteurs qui avaient déjé des coordonnées :", nombre_deja_geocodes)
print("Nouveaux acteurs géocodés par adresse précise :", nombre_geocodes_adresse)
print("Nouveaux acteurs géocodés par centre commune :", nombre_geocodes_commune)
print("échecs de géocodage :", nombre_echecs)
print("Acteurs toujours sans coordonnées :", len(df_sans_coordonnees))
print("\nFichiers créés :")
print("-", fichier_sortie)

  Coordonnées trouvées : 48.85635 2.403206
[21/1968] SAS CLEMENT / CIFC
  Géocodage adresse précise : Zone Industrielle d'Arbois 39600 Arbois
  Géocodage centre commune : 39600 Arbois
  Coordonnées trouvées : 46.8876 5.773
[22/1968] BE Geoxis Energétique SAS
  Géocodage adresse précise : 2 rue de la Gare 62144 Mont St Eloi
  Coordonnées trouvées : 50.343275 2.680422
[23/1968] SNCF MOBILITES - GARES & CONNEXIONS - AGENCE GARES CENTRE EST RHONE ALPIN
  Coordonn?es d?j? pr?sentes : page_source
[24/1968] HOPDURABLE
  Coordonn?es d?j? pr?sentes : page_source
[25/1968] KIT FORET
  Coordonn?es d?j? pr?sentes : page_source
[26/1968] NATUR'BOIS
  Coordonn?es d?j? pr?sentes : page_source
[27/1968] SCIERIE DULAC
  Coordonn?es d?j? pr?sentes : page_source
[28/1968] COMMUNE DE LANTRIAC
  Coordonn?es d?j? pr?sentes : page_source
[29/1968] ATELIER DAVID FARGETTE
  Coordonn?es d?j? pr?sentes : page_source
[30/1968] ROCHARD ET ASSOCIES
  Coordonn?es d?j? pr?sentes : page_source
[31/1968] AVP INGENIERIE

## 3.4 Geocodage international des acteurs sans coordonnees


In [11]:
# 1. Parametres

fichier_acteurs_geocodes = "acteurs_geocodes_complet.csv"

col_adresse = "adresse_detaillee"
col_commune = "commune"
col_lat = "latitude"
col_lon = "longitude"


# 2. Lire le fichier des acteurs geocodes

df = pd.read_csv(fichier_acteurs_geocodes, encoding="utf-8-sig", dtype={"code_postal": "string"})

if "commune" not in df.columns and "ville" in df.columns:
    df = df.rename(columns={"ville": "commune"})
elif "commune" in df.columns and "ville" in df.columns:
    masque_commune_vide = df["commune"].isna() | (df["commune"].astype(str).str.strip() == "")
    df.loc[masque_commune_vide, "commune"] = df.loc[masque_commune_vide, "ville"]
    df = df.drop(columns=["ville"])

if "commune" not in df.columns:
    df["commune"] = ""

df[col_lat] = pd.to_numeric(df[col_lat], errors="coerce")
df[col_lon] = pd.to_numeric(df[col_lon], errors="coerce")

if "adresse_geocodee" not in df.columns:
    df["adresse_geocodee"] = ""

if "source_geocodage" not in df.columns:
    df["source_geocodage"] = ""

if "source_coordonnees" not in df.columns:
    df["source_coordonnees"] = ""


# 3. Fonctions pour reconnaitre les communes internationales


def normaliser_pays_international(valeur):
    texte = nettoyer_texte(valeur).casefold()
    texte = texte.replace("-", " ")
    texte = re.sub(r"\s+", " ", texte).strip()
    return texte


def extraire_commune_internationale(valeur, code_postal="", adresse=""):
    texte = nettoyer_texte(valeur)
    code_postal_colonne = nettoyer_texte(code_postal)
    adresse = nettoyer_texte(adresse)

    if est_vide(texte) and est_vide(adresse):
        return "", "", ""

    if re.fullmatch(r"\d+\.0", code_postal_colonne):
        code_postal_colonne = code_postal_colonne[:-2]

    pays_par_nom = {
        "pays bas": "NL",
        "netherlands": "NL",
        "nederland": "NL",
        "belgique": "BE",
        "belgium": "BE",
        "allemagne": "DE",
        "germany": "DE",
        "suisse": "CH",
        "switzerland": "CH",
        "italie": "IT",
        "italy": "IT",
        "espagne": "ES",
        "spain": "ES",
        "luxembourg": "LU",
    }

    texte_recherche = " ".join(
        partie for partie in [adresse, texte] if not est_vide(partie)
    )

    match_autriche = re.search(r"\bA[\s-]+(\d{3,10})\b", texte_recherche, flags=re.IGNORECASE)
    if match_autriche:
        commune_autriche = re.sub(r"^[A-Za-z]{2}[\s-]+\d+(?:[\s-]+\d+)*\s+", "", texte).strip()
        if est_vide(commune_autriche):
            commune_autriche = re.sub(r".*\bA[\s-]+\d{3,10}\b", "", texte_recherche, flags=re.IGNORECASE).strip()
            commune_autriche = re.sub(r"^[A-Za-z]{2}[\s-]+\d+(?:[\s-]+\d+)*\s+", "", commune_autriche).strip()
        if commune_autriche:
            return "AT", match_autriche.group(1), commune_autriche

    match_code_pays_adresse = re.search(
        r"\b([A-Za-z]{2})[\s-]+(\d{3,10})\s+([^,()]+)",
        texte_recherche,
        flags=re.IGNORECASE,
    )
    if match_code_pays_adresse:
        code_pays = match_code_pays_adresse.group(1).upper()
        code_postal_trouve = match_code_pays_adresse.group(2)
        commune_trouvee = nettoyer_texte(match_code_pays_adresse.group(3))
        commune_trouvee = re.sub(r"\s+[A-Za-z]{2}[\s-]+\d.*$", "", commune_trouvee).strip()
        if code_pays and code_postal_trouve and commune_trouvee:
            return code_pays, code_postal_trouve, commune_trouvee

    villes_pays_par_defaut = {
        "barcelona": "ES",
        "barcelone": "ES",
    }
    pays_defaut = villes_pays_par_defaut.get(normaliser_pays_international(texte), "")
    if not pays_defaut:
        for ville, code_pays in villes_pays_par_defaut.items():
            if re.search(rf"\b{re.escape(ville)}\b", normaliser_pays_international(adresse)):
                pays_defaut = code_pays
                break
    if pays_defaut and code_postal_colonne:
        return pays_defaut, code_postal_colonne, texte

    if pays_defaut:
        match_code_ville = re.search(r"\b(\d{3,10})\s+([^,()]+)", adresse)
        if match_code_ville:
            code_postal_trouve = match_code_ville.group(1)
            commune_trouvee = texte or nettoyer_texte(match_code_ville.group(2))
            return pays_defaut, code_postal_trouve, commune_trouvee

    patterns = [
        # NL-6214 Maastricht, NL 6214 Maastricht, NL-6214, Maastricht
        r"^([A-Za-z]{2})[\s-]+(\d{3,10})\s*,?\s+(.+)$",
        # 6214 Maastricht, NL
        r"^(\d{3,10})\s+(.+?)\s*,\s*([A-Za-z]{2})$",
        # Pays-Bas - 6214 Maastricht
        r"^(.+?)\s+-\s+(\d{3,10})\s+(.+)$",
    ]

    for index, pattern in enumerate(patterns):
        match = re.match(pattern, texte, flags=re.IGNORECASE)
        if not match:
            continue

        if index == 0:
            code_pays = match.group(1).upper()
            code_postal = match.group(2)
            commune = nettoyer_texte(match.group(3))
        elif index == 1:
            code_postal = match.group(1)
            commune = nettoyer_texte(match.group(2))
            code_pays = match.group(3).upper()
        else:
            pays_texte = normaliser_pays_international(match.group(1))
            code_pays = pays_par_nom.get(pays_texte, "")
            code_postal = match.group(2)
            commune = nettoyer_texte(match.group(3))

        commune = re.sub(r"\s*,\s*[A-Za-z]{2}$", "", commune).strip()

        if code_pays and code_postal and commune:
            return code_pays, code_postal, commune

    return "", "", ""


def construire_requetes_internationales(adresse="", commune="", code_postal=""):
    code_pays, code_postal, commune_corrigee = extraire_commune_internationale(
        commune,
        code_postal=code_postal,
        adresse=adresse,
    )

    if not code_pays:
        return []

    pays_par_code = {
        "AT": "Austria",
        "BE": "Belgium",
        "CH": "Switzerland",
        "DE": "Germany",
        "DK": "Denmark",
        "ES": "Spain",
        "GB": "United Kingdom",
        "IT": "Italy",
        "LU": "Luxembourg",
        "NL": "Netherlands",
        "UK": "United Kingdom",
    }

    pays = pays_par_code.get(code_pays, code_pays)
    adresse = nettoyer_texte(adresse)
    commune_complete = f"{code_postal} {commune_corrigee}"

    requetes = []

    if not est_vide(adresse):
        requetes.append(f"{adresse}, {commune_complete}, {pays}")

    requetes.append(f"{commune_complete}, {pays}")
    requetes.append(f"{commune_corrigee}, {pays}")

    return list(dict.fromkeys(requetes))


def geocoder_adresse_internationale(adresse="", commune="", code_postal=""):
    requetes = construire_requetes_internationales(
        adresse=adresse,
        commune=commune,
        code_postal=code_postal,
    )

    if not requetes:
        return None, None, ""

    for requete in requetes:
        print("  Requete Nominatim :", requete)

        params = {
            "q": requete,
            "format": "json",
            "limit": 1,
            "addressdetails": 1,
        }

        try:
            reponse = requests.get(
                "https://nominatim.openstreetmap.org/search",
                params=params,
                headers={"User-Agent": "panorama-bois-geocoding-international/1.0"},
                timeout=20,
            )
            reponse.raise_for_status()
            data = reponse.json()

            if not data:
                time.sleep(1)
                continue

            resultat = data[0]
            latitude = float(resultat["lat"])
            longitude = float(resultat["lon"])
            label = resultat.get("display_name", requete)

            return latitude, longitude, label

        except Exception as e:
            print("  Erreur geocodage international :", requete, "|", e)
            time.sleep(1)

    return None, None, ""


# 4. Completer seulement les acteurs qui n'ont pas encore de coordonnees

masque_sans_coordonnees = df[col_lat].isna() | df[col_lon].isna()
indices_a_traiter = df[masque_sans_coordonnees].index.tolist()

nombre_total = len(indices_a_traiter)
nombre_geocodes_international = 0
nombre_non_reconnus = 0
nombre_echecs = 0

for compteur, index in enumerate(indices_a_traiter, start=1):
    row = df.loc[index]
    adresse = nettoyer_texte(row.get(col_adresse, ""))
    commune = nettoyer_texte(row.get(col_commune, ""))
    code_postal = nettoyer_texte(row.get(col_code_postal, ""))

    print(f"[{compteur}/{nombre_total}] Geocodage international : {commune}")

    if not extraire_commune_internationale(commune, code_postal=code_postal, adresse=adresse)[0]:
        nombre_non_reconnus += 1
        print("  Format international non reconnu")
        continue

    latitude, longitude, adresse_geocodee = geocoder_adresse_internationale(
        adresse=adresse,
        commune=commune,
        code_postal=code_postal,
    )

    if latitude is not None and longitude is not None:
        df.at[index, col_lat] = latitude
        df.at[index, col_lon] = longitude
        df.at[index, "adresse_geocodee"] = adresse_geocodee
        df.at[index, "source_geocodage"] = "geocodage_international"
        df.at[index, "source_coordonnees"] = "geocodage_international"
        nombre_geocodes_international += 1
        print("  Coordonnees trouvees :", latitude, longitude)
    else:
        nombre_echecs += 1
        print("  Coordonnees non trouvees")

    time.sleep(1)


# 5. Sauvegarder le fichier complete

df.to_csv(fichier_acteurs_geocodes, index=False, encoding="utf-8-sig")

print("\nGeocodage international termine.")
print("Acteurs sans coordonnees au depart :", nombre_total)
print("Formats internationaux non reconnus :", nombre_non_reconnus)
print("Acteurs geocodes par adresse internationale :", nombre_geocodes_international)
print("Echecs de geocodage international :", nombre_echecs)
print("Fichier mis a jour :")
print("-", fichier_acteurs_geocodes)


[1/891] Geocodage international : 
  Format international non reconnu
[2/891] Geocodage international : NL-6214 Maastricht
  Requete Nominatim : Volkspein 33 (blue port) NL-6214 Maastricht, 6214 Maastricht, Netherlands
  Requete Nominatim : 6214 Maastricht, Netherlands
  Coordonnees trouvees : 50.8512438 5.6909768
[3/891] Geocodage international : La Plaine Saint-Denis
  Format international non reconnu
[4/891] Geocodage international : DE-88436 EBERHARDZELL
  Requete Nominatim : DE-88436 EBERHARDZELL, 88436 EBERHARDZELL, Germany
  Requete Nominatim : 88436 EBERHARDZELL, Germany
  Coordonnees trouvees : 47.9905687 9.8335007
[5/891] Geocodage international : DK-8700 Horsens
  Requete Nominatim : Bygholm Søpark 23 DK-8700 Horsens (45)70110200, 8700 Horsens, Denmark
  Requete Nominatim : 8700 Horsens, Denmark
  Coordonnees trouvees : 55.8623835 9.8448761
[6/891] Geocodage international : Barcelona
  Requete Nominatim : C. Quintana 4 08002 Barcelona (34)933173188, 08002 Barcelona, Spain
  

## 3.5 Fusion des acteurs et des coordonnees


In [ ]:
# 1. Paramétres

base_site = "https://www.panoramabois.fr"

fichier_acteurs_projets = "acteurs_projets.csv"
fichier_acteurs_geocodes = "acteurs_geocodes_complet.csv"
fichier_fusion = "acteurs_projets_geocodes.csv"


# 2. Lire les fichiers CSV

df_projets = pd.read_csv(fichier_acteurs_projets, encoding="utf-8-sig", dtype=str)
df_geocodes = pd.read_csv(fichier_acteurs_geocodes, encoding="utf-8-sig", dtype=str)

if "commune" not in df_geocodes.columns and "ville" in df_geocodes.columns:
    df_geocodes = df_geocodes.rename(columns={"ville": "commune"})
elif "commune" in df_geocodes.columns and "ville" in df_geocodes.columns:
    masque_commune_vide = df_geocodes["commune"].isna() | (df_geocodes["commune"].astype(str).str.strip() == "")
    df_geocodes.loc[masque_commune_vide, "commune"] = df_geocodes.loc[masque_commune_vide, "ville"]
    df_geocodes = df_geocodes.drop(columns=["ville"])

print("Colonnes de acteurs_projets :")
print(df_projets.columns.tolist())

print("\nColonnes de acteurs_geocodes_complet :")
print(df_geocodes.columns.tolist())


# 3. Colonnes standard des CSV créés par ce notebook

# acteurs_projets.csv
col_role_projets = "role_acteur"
col_groupe_projets = "groupe_acteur"
col_nom_projets = "acteur_nom"
col_url_acteur_projets = "acteur_url"
col_id_acteur_projets = None  # Identifiant extrait de acteur_url
col_nom_projet = None  # Non présent dans acteurs_projets.csv
col_id_projet = "projet_id"
col_url_projet = "url_projet"

# acteurs_geocodes_complet.csv
col_nom_geocodes = "nom_acteur"
col_url_acteur_geocodes = "url_acteur"
col_id_acteur_geocodes = "id_acteur"
col_lat = "latitude"
col_lon = "longitude"
col_adresse = "adresse_detaillee"
col_commune = "commune"
col_code_postal = "code_postal"


def verifier_colonnes(df, colonnes, nom_fichier):
    colonnes_manquantes = [colonne for colonne in colonnes if colonne not in df.columns]

    if colonnes_manquantes:
        raise ValueError(
            f"Colonnes manquantes dans {nom_fichier} : {colonnes_manquantes}"
        )


verifier_colonnes(
    df_projets,
    [
        col_role_projets,
        col_groupe_projets,
        col_nom_projets,
        col_url_acteur_projets,
        col_id_projet,
        col_url_projet,
    ],
    fichier_acteurs_projets,
)

verifier_colonnes(
    df_geocodes,
    [
        col_nom_geocodes,
        col_url_acteur_geocodes,
        col_id_acteur_geocodes,
        col_lat,
        col_lon,
        col_adresse,
        col_commune,
        col_code_postal,
    ],
    fichier_acteurs_geocodes,
)


# 4. Fonctions de nettoyage et de clé acteur


def normaliser_nom(valeur):
    return nettoyer_texte(valeur).casefold()


def normaliser_url(valeur):
    if est_vide(valeur):
        return ""

    return urljoin(base_site, nettoyer_texte(valeur)).rstrip("/")


def extraire_id_acteur(valeur):
    if est_vide(valeur):
        return ""

    texte = nettoyer_texte(valeur)
    match = re.search(r"/acteurs/(\d+)", texte)

    if match:
        return match.group(1)

    if texte.isdigit():
        return texte

    return ""


def construire_cle_acteur(id_acteur="", url_acteur="", nom_acteur=""):
    id_acteur = extraire_id_acteur(id_acteur)
    url_acteur = normaliser_url(url_acteur)
    nom_acteur = normaliser_nom(nom_acteur)

    if id_acteur:
        return f"id:{id_acteur}"

    id_depuis_url = extraire_id_acteur(url_acteur)

    if id_depuis_url:
        return f"id:{id_depuis_url}"

    if url_acteur:
        return f"url:{url_acteur}"

    if nom_acteur:
        return f"nom:{nom_acteur}"

    return ""


def simplifier_role_acteur(role):
    role = nettoyer_texte(role)
    if not role:
        return "R?le non renseign?"

    role = role.replace("?", "'").replace("?", "'")
    role = re.sub(r"\s+", " ", role).strip()
    role = re.sub(r"\s+-\s+.*$", "", role).strip()
    role = role.replace("d'oeuvre", "d'?uvre")

    return role


def simplifier_groupe_acteur(groupe):
    groupe = nettoyer_texte(groupe)
    if not groupe:
        return "Groupe non renseign?"

    groupe = groupe.replace("?", "'").replace("?", "'")
    groupe = re.sub(r"\s+", " ", groupe).strip()
    groupe = re.sub(r"\s*\(\d+\)\s*$", "", groupe).strip()
    groupe = groupe.replace("d'oeuvre", "d'?uvre")

    return groupe


# 5. Préparer acteurs_projets

df_projets["nom_acteur_fusion"] = df_projets[col_nom_projets].apply(nettoyer_texte)

df_projets["groupe_acteur_fusion"] = df_projets[col_groupe_projets].apply(nettoyer_texte)
df_projets["groupe_acteur_carte"] = df_projets["groupe_acteur_fusion"].apply(
    simplifier_groupe_acteur
)

df_projets["role_acteur_fusion"] = df_projets[col_role_projets].apply(nettoyer_texte)
df_projets["role_acteur_fusion"] = df_projets["role_acteur_fusion"].replace(
    "", "Réle non renseigné"
)
df_projets["role_acteur_carte"] = df_projets["role_acteur_fusion"].apply(
    simplifier_role_acteur
)

df_projets["url_acteur_fusion"] = df_projets[col_url_acteur_projets].apply(
    normaliser_url
)

df_projets["id_acteur_fusion"] = df_projets["url_acteur_fusion"].apply(
    extraire_id_acteur
)

if col_id_acteur_projets:
    df_projets["id_acteur_fusion"] = df_projets[col_id_acteur_projets].apply(
        extraire_id_acteur
    )

df_projets["cle_acteur"] = df_projets.apply(
    lambda row: construire_cle_acteur(
        id_acteur=row["id_acteur_fusion"],
        url_acteur=row["url_acteur_fusion"],
        nom_acteur=row["nom_acteur_fusion"],
    ),
    axis=1,
)

if col_nom_projet:
    df_projets["nom_projet_fusion"] = df_projets[col_nom_projet].apply(nettoyer_texte)
else:
    df_projets["nom_projet_fusion"] = ""

if col_id_projet:
    df_projets["id_projet_fusion"] = df_projets[col_id_projet].apply(nettoyer_texte)
else:
    df_projets["id_projet_fusion"] = ""

if col_url_projet:
    df_projets["url_projet_fusion"] = df_projets[col_url_projet].apply(normaliser_url)
else:
    df_projets["url_projet_fusion"] = ""


# 6. Préparer acteurs_geocodes

df_geocodes["nom_acteur_geo"] = df_geocodes[col_nom_geocodes].apply(nettoyer_texte)

df_geocodes["url_acteur_geo"] = df_geocodes[col_url_acteur_geocodes].apply(
    normaliser_url
)

df_geocodes["id_acteur_geo"] = df_geocodes[col_id_acteur_geocodes].apply(
    extraire_id_acteur
)

df_geocodes["cle_acteur"] = df_geocodes.apply(
    lambda row: construire_cle_acteur(
        id_acteur=row["id_acteur_geo"],
        url_acteur=row["url_acteur_geo"],
        nom_acteur=row["nom_acteur_geo"],
    ),
    axis=1,
)

df_geocodes[col_lat] = pd.to_numeric(df_geocodes[col_lat], errors="coerce")
df_geocodes[col_lon] = pd.to_numeric(df_geocodes[col_lon], errors="coerce")

if "source_coordonnees" not in df_geocodes.columns:
    df_geocodes["source_coordonnees"] = ""

df_geocodes["adresse_acteur"] = df_geocodes[col_adresse].apply(nettoyer_texte)
df_geocodes["commune_acteur"] = df_geocodes[col_commune].apply(nettoyer_texte)
df_geocodes["code_postal_acteur"] = df_geocodes[col_code_postal].apply(
    normaliser_code_postal
)

masque_coordonnees_sans_localisation = (
    df_geocodes[col_lat].notna()
    & df_geocodes[col_lon].notna()
    & df_geocodes["adresse_acteur"].apply(est_vide)
    & df_geocodes["commune_acteur"].apply(est_vide)
    & df_geocodes["code_postal_acteur"].apply(est_vide)
)

if masque_coordonnees_sans_localisation.any():
    print(
        "Coordonnées ignorées sans localisation exploitable :",
        int(masque_coordonnees_sans_localisation.sum()),
    )
    df_geocodes.loc[masque_coordonnees_sans_localisation, [col_lat, col_lon]] = pd.NA
    df_geocodes.loc[masque_coordonnees_sans_localisation, "source_coordonnees"] = ""

# Ne garder qu'une ligne géographique par acteur, identifiée par id_acteur

df_geocodes = df_geocodes[df_geocodes["id_acteur_geo"] != ""].copy()

df_geocodes_unique = df_geocodes.sort_values(
    by=[col_lat, col_lon], na_position="last"
).drop_duplicates(subset=["id_acteur_geo"], keep="first")


# 7. Fusionner les deux fichiers

colonnes_geocodes_a_garder = [
    "nom_acteur_geo",
    "url_acteur_geo",
    "id_acteur_geo",
    col_lat,
    col_lon,
    "adresse_acteur",
    "commune_acteur",
    "code_postal_acteur",
    "source_coordonnees",
]

df_fusion = df_projets.merge(
    df_geocodes_unique[colonnes_geocodes_a_garder],
    left_on="id_acteur_fusion",
    right_on="id_acteur_geo",
    how="left",
    validate="many_to_one",
)

df_fusion["nom_acteur_final"] = df_fusion["nom_acteur_fusion"].where(
    df_fusion["nom_acteur_fusion"] != "", df_fusion["nom_acteur_geo"]
)

df_fusion["url_acteur_final"] = df_fusion["url_acteur_fusion"].where(
    df_fusion["url_acteur_fusion"] != "", df_fusion["url_acteur_geo"]
)


# 8. Vérifier, sauvegarder et résumer la fusion

nombre_lignes = len(df_fusion)
nombre_geocodes_source = df_geocodes[
    df_geocodes[col_lat].notna() & df_geocodes[col_lon].notna()
].shape[0]
nombre_acteurs_fusionnes = df_fusion["id_acteur_geo"].notna().sum()
nombre_avec_coordonnees = df_fusion[
    df_fusion[col_lat].notna() & df_fusion[col_lon].notna()
].shape[0]
nombre_sans_coordonnees = nombre_lignes - nombre_avec_coordonnees

if nombre_geocodes_source > 0 and nombre_acteurs_fusionnes == 0:
    raise ValueError(
        "Fusion invalide : aucune ligne de acteurs_projets.csv ne correspond "
        "aux acteurs géocodés. Vérifier cle_acteur, acteur_url et id_acteur."
    )

if nombre_geocodes_source > 0 and nombre_avec_coordonnees == 0:
    raise ValueError(
        "Fusion invalide : acteurs_geocodes_complet.csv contient des coordonnées, "
        "mais aucune coordonnée n'a été transférée dans la fusion."
    )

df_fusion.to_csv(fichier_fusion, index=False, encoding="utf-8-sig")

print("\nRésultat de la fusion :")
print("Nombre total de lignes :", nombre_lignes)
print("Acteurs géocodés avec coordonnées en source :", nombre_geocodes_source)
print("Lignes ayant trouvé un acteur géocodé :", nombre_acteurs_fusionnes)
print("Lignes avec coordonnées :", nombre_avec_coordonnees)
print("Lignes sans coordonnées :", nombre_sans_coordonnees)
print("\nFichier créé :")
print("-", fichier_fusion)


## 3.6 Affichage des acteurs sur la carte


In [ ]:
from html import escape as echapper_html_std
from folium.plugins import MarkerCluster, TreeLayerControl

# 1. Paramétres

fichier_fusion = "acteurs_projets_geocodes.csv"
fichier_carte = "carte_acteurs_par_role.html"

col_lat = "latitude"
col_lon = "longitude"


# 2. Lire le fichier fusionné

df = pd.read_csv(fichier_fusion, encoding="utf-8-sig", dtype=str)


def echapper_html(valeur):
    return echapper_html_std(nettoyer_texte(valeur))


# 3. Ne garder que les acteurs cartographiables

df_map = df.dropna(subset=[col_lat, col_lon]).copy()

df_map[col_lat] = pd.to_numeric(df_map[col_lat], errors="coerce")
df_map[col_lon] = pd.to_numeric(df_map[col_lon], errors="coerce")

df_map = df_map.dropna(subset=[col_lat, col_lon]).copy()

colonnes_localisation = ["adresse_acteur", "commune_acteur", "code_postal_acteur"]
if all(colonne in df_map.columns for colonne in colonnes_localisation):
    masque_localisation_valide = ~(
        df_map["adresse_acteur"].apply(est_vide)
        & df_map["commune_acteur"].apply(est_vide)
        & df_map["code_postal_acteur"].apply(est_vide)
    )
    df_map = df_map[masque_localisation_valide].copy()


def normaliser_groupe_carte(groupe):
    groupe = nettoyer_texte(groupe)
    groupe = re.sub(r"\s*\(\d+\)\s*$", "", groupe).strip()
    return groupe or "Groupe non renseigné"


df_map["groupe_acteur_carte"] = df_map["groupe_acteur_carte"].apply(
    normaliser_groupe_carte
)

# Vérification des coordonnées franéaises approximatives
df_map = df_map[
    df_map[col_lat].between(41, 52) & df_map[col_lon].between(-6, 10)
].copy()

if df_map.empty:
    raise ValueError("Aucun acteur ne posséde de coordonnées valides.")


# 4. Agréger les projets par acteur et réle


def valeurs_uniques(serie):
    valeurs = []

    for valeur in serie:
        valeur = nettoyer_texte(valeur)

        if valeur and valeur not in valeurs:
            valeurs.append(valeur)

    return valeurs


colonnes_groupement = [
    "cle_acteur",
    "groupe_acteur_carte",
    "role_acteur_carte",
    "nom_acteur_final",
    "url_acteur_final",
    col_lat,
    col_lon,
    "adresse_acteur",
    "commune_acteur",
    "code_postal_acteur",
]


df_acteurs_roles = df_map.groupby(
    colonnes_groupement, dropna=False, as_index=False
).agg(
    {
        "nom_projet_fusion": valeurs_uniques,
        "id_projet_fusion": valeurs_uniques,
        "url_projet_fusion": valeurs_uniques,
    }
)


# 5. Créer la carte

centre_lat = df_acteurs_roles[col_lat].mean()
centre_lon = df_acteurs_roles[col_lon].mean()

m = folium.Map(
    location=[centre_lat, centre_lon],
    zoom_start=6,
    tiles="OpenStreetMap",
    control_scale=True,
)


# 6. Définir une couleur par réle

couleurs_folium = [
    "blue",
    "red",
    "green",
    "purple",
    "orange",
    "darkred",
    "lightred",
    "beige",
    "darkblue",
    "darkgreen",
    "cadetblue",
    "darkpurple",
    "pink",
    "lightblue",
    "lightgreen",
    "gray",
    "black",
    "lightgray",
]

groupes_roles = (
    df_acteurs_roles[["groupe_acteur_carte", "role_acteur_carte"]]
    .dropna()
    .drop_duplicates()
    .sort_values(["groupe_acteur_carte", "role_acteur_carte"])
)

cles_couleur = [
    (row["groupe_acteur_carte"], row["role_acteur_carte"])
    for _, row in groupes_roles.iterrows()
]

couleur_par_role = {
    cle: couleurs_folium[index % len(couleurs_folium)]
    for index, cle in enumerate(cles_couleur)
}


# 7. Créer un arbre de calques par groupe d'acteur et par rôle

arbre_calques = []
calques_par_groupe = {}

for groupe_acteur, role in cles_couleur:
    df_role = df_acteurs_roles[
        (df_acteurs_roles["groupe_acteur_carte"] == groupe_acteur)
        & (df_acteurs_roles["role_acteur_carte"] == role)
    ].copy()

    nom_calque = f"{groupe_acteur} / {role} ({len(df_role)})"
    groupe_role = folium.FeatureGroup(name=nom_calque, show=True).add_to(m)
    calques_par_groupe.setdefault(groupe_acteur, []).append(
        {"label": f"{role} ({len(df_role)})", "layer": groupe_role}
    )
    cluster_role = MarkerCluster(name=f"{groupe_acteur} / {role}").add_to(groupe_role)
    couleur = couleur_par_role[(groupe_acteur, role)]

    for _, row in df_role.iterrows():
        nom_acteur = nettoyer_texte(row["nom_acteur_final"])
        adresse = nettoyer_texte(row["adresse_acteur"])
        commune = nettoyer_texte(row["commune_acteur"])
        code_postal = normaliser_code_postal(row["code_postal_acteur"])
        latitude = row[col_lat]
        longitude = row[col_lon]

        projets = row["nom_projet_fusion"]
        ids_projets = row["id_projet_fusion"]
        urls_projets = row["url_projet_fusion"]

        lignes_projets = []
        for numero, nom_projet in enumerate(projets):
            id_projet = ids_projets[numero] if numero < len(ids_projets) else ""
            url_projet = urls_projets[numero] if numero < len(urls_projets) else ""

            libelle_projet = nom_projet or id_projet or url_projet
            if not libelle_projet:
                continue

            ligne = f"<li>{echapper_html(libelle_projet)}</li>"
            lignes_projets.append(ligne)

        if lignes_projets:
            projets_html = (
                "<ul style='padding-left:18px;'>" + "".join(lignes_projets) + "</ul>"
            )
        else:
            projets_html = "Aucun projet renseigné"

        popup_html = f"""
        <div style="width:340px;">
            <h4 style="margin-bottom:8px;">
                {echapper_html(nom_acteur)}
            </h4>

            <b>Réle :</b>
            {echapper_html(role)}
            <br>

            <b>Adresse :</b>
            {echapper_html(adresse)}
            <br>

            <b>Commune :</b>
            {echapper_html(commune)}
            <br>

            <b>Code postal :</b>
            {echapper_html(code_postal)}
            <br>

            <b>Coordonnées :</b>
            {latitude:.6f}, {longitude:.6f}
            <br><br>

            <b>Projets associés :</b>
            {projets_html}
        </div>
        """

        folium.Marker(
            location=[latitude, longitude],
            popup=folium.Popup(popup_html, max_width=420),
            tooltip=(f"{nom_acteur} é {role}"),
            icon=folium.Icon(color=couleur, icon="building", prefix="fa"),
        ).add_to(cluster_role)


for groupe_acteur in sorted(calques_par_groupe):
    arbre_calques.append(
        {
            "label": f"<b>{echapper_html(groupe_acteur)}</b>",
            "select_all_checkbox": True,
            "children": calques_par_groupe[groupe_acteur],
        }
    )


# 8. Adapter l'étendue de la carte

coordonnees = df_acteurs_roles[[col_lat, col_lon]].values.tolist()

if len(coordonnees) > 1:
    m.fit_bounds(coordonnees)


# 9. Ajouter le contrôle hiérarchique des couches et sauvegarder

TreeLayerControl(
    overlay_tree=arbre_calques,
    collapsed=False,
    named_toggle=True,
    selector_back=True,
    label_is_selector="both",
    collapse_all="Tout replier",
    expand_all="Tout afficher",
).add_to(m)
m.save(fichier_carte)

resume_roles = (
    df_acteurs_roles.groupby(["groupe_acteur_carte", "role_acteur_carte"])
    .size()
    .reset_index(name="nombre_acteurs")
    .sort_values("nombre_acteurs", ascending=False)
)

print("\nNombre d'acteurs par réle :")
display(resume_roles)

print("\nFichier créé :")
print("-", fichier_carte)

m


## 4.1 Informations techniques des projets


In [ ]:
# Parametres de l'extraction technique
FICHIER_PROJETS = "liens_projets.csv"
COLONNE_LIEN = "url_projet"

CONSERVER_LIGNES_VIDES = False
SAUVEGARDE_TOUS_LES = 25

FICHIER_PROJETS_TECHNIQUES = "projets_infos_techniques.csv"
FICHIER_PRODUITS_TECHNIQUES = "projets_produits_techniques.csv"


In [ ]:

# Fonctions de nettoyage et de parsing



def nettoyer_texte_multiligne(texte):
    """
    Nettoie un texte long tout en conservant
    les retours à la ligne significatifs.
    """

    if texte is None:
        return ""

    texte = str(texte).replace("\xa0", " ")

    lignes_resultat = []

    for ligne in texte.splitlines():

        ligne = re.sub(r"[ \t]+", " ", ligne).strip()

        if ligne:
            lignes_resultat.append(ligne)

    return "\n".join(lignes_resultat)



# 4. Extraction du numéro du projet



def extraire_numero_projet(url):
    """
    Extrait le numéro du projet depuis son URL.

    Exemple :
    https://www.panoramabois.fr/projets/3828

    Résultat :
    3828
    """

    match = re.search(r"/projets/(\d+)", str(url))

    if match:
        return match.group(1)

    return ""


def charger_soup_projet_technique(url):
    soup, _ = charger_page_projet(url)

    return soup



# Colonnes uniques pour les tableaux



def rendre_colonnes_uniques(colonnes):
    resultat = []
    compteurs = {}

    for position, colonne in enumerate(colonnes, start=1):

        colonne = nettoyer_texte(colonne)

        if not colonne:
            colonne = f"colonne_{position}"

        compteurs[colonne] = compteurs.get(colonne, 0) + 1

        if compteurs[colonne] == 1:
            resultat.append(colonne)

        else:
            resultat.append(f"{colonne}_{compteurs[colonne]}")

    return resultat



# Extraction d'une valeur dans un paragraphe
def extraire_valeur_dans_paragraphe(paragraphe, balise_label):
    morceaux = []

    for voisin in balise_label.next_siblings:

        if isinstance(voisin, NavigableString):

            texte = str(voisin).replace("\xa0", " ")

            if texte.strip():
                morceaux.append(texte.strip())

        elif isinstance(voisin, Tag):

            if voisin.name == "br":
                morceaux.append("\n")

            else:
                texte = voisin.get_text("\n", strip=True)

                if texte:
                    morceaux.append(texte)

    texte_final = "\n".join(morceaux)

    return nettoyer_texte_multiligne(texte_final)



# 9. Extraction d'une valeur après strong ou b
def extraire_valeur_apres_balise(balise_label):

    morceaux = []

    for voisin in balise_label.next_siblings:

        if isinstance(voisin, NavigableString):

            texte = nettoyer_texte(voisin)

            if texte:
                morceaux.append(texte)

        elif isinstance(voisin, Tag):

            if voisin.name in {"br", "h2", "h3", "p", "div", "table", "strong", "b"}:
                break

            texte = nettoyer_texte(voisin)

            if texte:
                morceaux.append(texte)

    return nettoyer_texte(" ".join(morceaux))



# Extraction d'un tableau HTML
def extraire_un_tableau(
    tableau,
    numero_projet,
    nom_projet,
    url,
    categorie_tab2,
    rubrique,
    sous_rubrique,
    numero_tableau,
    ordre_element,
):

    resultats = []

    lignes = tableau.find_all("tr")

    if not lignes:
        return resultats

    
    # Recherche des en-têtes
    entetes_brutes = []

    for ligne in lignes:

        cellules_th = ligne.find_all("th", recursive=False)

        if cellules_th:

            entetes_brutes = [nettoyer_texte(cellule) for cellule in cellules_th]

            break

    
    # Aucun en-tête trouvé
    

    if not entetes_brutes:

        nombre_colonnes = max(
            [len(ligne.find_all(["th", "td"], recursive=False)) for ligne in lignes],
            default=0,
        )

        entetes_brutes = [
            f"colonne_{position}" for position in range(1, nombre_colonnes + 1)
        ]

    entetes = rendre_colonnes_uniques(entetes_brutes)

    
    # Extraction des lignes td
    

    numero_ligne = 0

    for ligne in lignes:

        cellules_td = ligne.find_all("td", recursive=False)

        if not cellules_td:
            continue

        valeurs = [nettoyer_texte(cellule) for cellule in cellules_td]

        ligne_entierement_vide = not any(valeur.strip() for valeur in valeurs)

        if ligne_entierement_vide and not CONSERVER_LIGNES_VIDES:
            continue

        numero_ligne += 1

        # Ajouter des colonnes supplémentaires
        while len(entetes) < len(valeurs):

            nouvelle_colonne = f"colonne_{len(entetes) + 1}"

            entetes.append(nouvelle_colonne)

        # Compléter les valeurs absentes
        while len(valeurs) < len(entetes):
            valeurs.append("")

        resultat_ligne = {
            "numero_projet": numero_projet,
            "nom_projet": nom_projet,
            "lien": url,
            "categorie_tab2": categorie_tab2,
            "rubrique": rubrique,
            "sous_rubrique": sous_rubrique,
            "ordre_element": ordre_element,
            "numero_tableau": numero_tableau,
            "numero_ligne": numero_ligne,
            "ligne_vide": ligne_entierement_vide,
        }

        for colonne, valeur in zip(entetes, valeurs):
            resultat_ligne[colonne] = valeur

        resultats.append(resultat_ligne)

    return resultats



# Extraction du texte complet de chaque section h3
def extraire_sections_tab2(tab2, numero_projet, nom_projet, url, categorie_tab2):

    sections = []

    titres_h3 = tab2.find_all("h3", recursive=False)

    for numero_section, titre_h3 in enumerate(titres_h3, start=1):

        rubrique = nettoyer_texte(titre_h3)

        morceaux = []

        for element in titre_h3.next_siblings:

            if isinstance(element, Tag) and element.name == "h3":
                break

            if isinstance(element, NavigableString):

                texte = nettoyer_texte(element)

                if texte:
                    morceaux.append(texte)

            elif isinstance(element, Tag):

                texte = nettoyer_texte(element)

                if texte:
                    morceaux.append(texte)

        sections.append(
            {
                "numero_projet": numero_projet,
                "nom_projet": nom_projet,
                "lien": url,
                "categorie_tab2": categorie_tab2,
                "numero_section": numero_section,
                "rubrique": rubrique,
                "texte_section": nettoyer_texte(" ".join(morceaux)),
            }
        )

    return sections



# 12. Extraction complète de tab-2
def extraire_tab2(soup, url):

    numero_projet = extraire_numero_projet(url)

    nom_projet = nettoyer_texte(soup.find("h1", class_="Project-title"))

    tab2 = soup.find("div", id="tab-2")

    resultat = {
        "numero_projet": numero_projet,
        "nom_projet": nom_projet,
        "lien": url,
        "categorie_tab2": "",
        "texte_complet": "",
        "champs": [],
        "tableaux": [],
        "sections": [],
    }

    if tab2 is None:
        return resultat

    
    # Catégorie générale
    titre_h2 = tab2.find("h2", recursive=False)

    if titre_h2 is None:
        titre_h2 = tab2.find("h2")

    if titre_h2:
        resultat["categorie_tab2"] = nettoyer_texte(titre_h2)

    
    # Texte complet de tab-2
    resultat["texte_complet"] = nettoyer_texte(tab2)

    
    # Sections h3 complètes
    resultat["sections"] = extraire_sections_tab2(
        tab2=tab2,
        numero_projet=numero_projet,
        nom_projet=nom_projet,
        url=url,
        categorie_tab2=resultat["categorie_tab2"],
    )

    
    # Variables permettant de conserver le contexte
    rubrique_actuelle = ""
    sous_rubrique_actuelle = ""

    numero_tableau = 0
    ordre_element = 0

    
    # Lecture dans l'ordre du HTML
    for element in tab2.children:

        if not isinstance(element, Tag):
            continue

        ordre_element += 1

        
        # Titre principal h2
        if element.name == "h2":
            continue

        
        # Nouvelle rubrique h3
        elif element.name == "h3":

            rubrique_actuelle = nettoyer_texte(element)

            sous_rubrique_actuelle = ""

        
        # Paragraphe contenant strong ou b
        elif element.name == "p":

            balise_label = element.find(["strong", "b"], recursive=False)

            if balise_label is None:
                continue

            champ = nettoyer_texte(balise_label)

            champ = re.sub(r"\s*:\s*$", "", champ).strip()

            valeur = extraire_valeur_dans_paragraphe(element, balise_label)

            if valeur:

                resultat["champs"].append(
                    {
                        "numero_projet": numero_projet,
                        "nom_projet": nom_projet,
                        "lien": url,
                        "categorie_tab2": resultat["categorie_tab2"],
                        "rubrique": rubrique_actuelle,
                        "sous_rubrique": "",
                        "ordre_element": ordre_element,
                        "champ": champ,
                        "valeur": valeur,
                        "type_valeur": "texte_long",
                    }
                )

                sous_rubrique_actuelle = ""

            else:
                sous_rubrique_actuelle = champ

        
        # Strong ou b directement dans tab-2
        elif element.name in {"strong", "b"}:

            champ = nettoyer_texte(element)

            champ = re.sub(r"\s*:\s*$", "", champ).strip()

            valeur = extraire_valeur_apres_balise(element)

            if champ:

                resultat["champs"].append(
                    {
                        "numero_projet": numero_projet,
                        "nom_projet": nom_projet,
                        "lien": url,
                        "categorie_tab2": resultat["categorie_tab2"],
                        "rubrique": rubrique_actuelle,
                        "sous_rubrique": (sous_rubrique_actuelle),
                        "ordre_element": ordre_element,
                        "champ": champ,
                        "valeur": valeur,
                        "type_valeur": "champ_simple",
                    }
                )

        
        # Tableau directement dans tab-2
        elif element.name == "table":

            numero_tableau += 1

            lignes_tableau = extraire_un_tableau(
                tableau=element,
                numero_projet=numero_projet,
                nom_projet=nom_projet,
                url=url,
                categorie_tab2=resultat["categorie_tab2"],
                rubrique=rubrique_actuelle,
                sous_rubrique=(sous_rubrique_actuelle),
                numero_tableau=numero_tableau,
                ordre_element=ordre_element,
            )

            resultat["tableaux"].extend(lignes_tableau)

            sous_rubrique_actuelle = ""

        
        # Div contenant un ou plusieurs tableaux
        elif element.name == "div":

            tableaux = element.find_all("table")

            if not tableaux:
                continue

            for tableau in tableaux:

                numero_tableau += 1

                lignes_tableau = extraire_un_tableau(
                    tableau=tableau,
                    numero_projet=numero_projet,
                    nom_projet=nom_projet,
                    url=url,
                    categorie_tab2=resultat["categorie_tab2"],
                    rubrique=rubrique_actuelle,
                    sous_rubrique=(sous_rubrique_actuelle),
                    numero_tableau=(numero_tableau),
                    ordre_element=ordre_element,
                )

                resultat["tableaux"].extend(lignes_tableau)

            sous_rubrique_actuelle = ""

    return resultat



# 13. Sauvegarde des résultats
# Standardisation des resultats techniques

import unicodedata

COLONNES_PROJETS_TECHNIQUES = [
    "id_projet",
    "type_batiment",
    "annee_livraison",
    "surface_m2",
    "cout_total_eur",
    "volume_lot_bois_m3",
    "cout_lot_bois_eur",
    "isolation_murs_mm",
    "approche_environnementale",
    "chauffage_bois",
    "appareil_chauffage_bois",
]

COLONNES_PRODUITS_TECHNIQUES = [
    "id_projet",
    "categorie",
    "sous_categorie",
    "produit",
    "essence",
    "provenance",
]


def normaliser_cle_technique(valeur):
    texte = nettoyer_texte(valeur).casefold()
    texte = unicodedata.normalize("NFKD", texte)
    texte = "".join(
        caractere for caractere in texte if not unicodedata.combining(caractere)
    )
    texte = re.sub(r"[^a-z0-9]+", "_", texte)
    return texte.strip("_")


def extraire_nombre_technique(valeur):
    texte = nettoyer_texte(valeur)

    if not texte:
        return ""

    match = re.search(r"[-+]?\d[\d\s.,]*", texte)

    if not match:
        return ""

    nombre = match.group(0).replace(" ", "")

    if "," in nombre and "." in nombre:
        nombre = nombre.replace(".", "").replace(",", ".")
    else:
        nombre = nombre.replace(",", ".")

    return nombre


def valeur_champ_par_mots_cles(champs, groupes_mots_cles):
    champs_indexes = []

    for champ in champs:
        contexte = " ".join(
            [
                champ.get("rubrique", ""),
                champ.get("sous_rubrique", ""),
                champ.get("champ", ""),
            ]
        )
        champs_indexes.append((normaliser_cle_technique(contexte), champ))

    for mots_cles in groupes_mots_cles:
        for cle, champ in champs_indexes:
            if all(mot in cle for mot in mots_cles):
                return nettoyer_texte(champ.get("valeur", ""))

    return ""


def extraire_projet_technique(resultat):
    champs = resultat.get("champs", [])

    ligne = {
        "id_projet": resultat.get("numero_projet", ""),
        "type_batiment": valeur_champ_par_mots_cles(
            champs, [["type", "batiment"], ["typologie"], ["batiment"]]
        ),
        "annee_livraison": extraire_nombre_technique(
            valeur_champ_par_mots_cles(
                champs, [
                    ["annee", "realisation"],
                    ["annee", "livraison"],
                    ["livraison"],
                    ["date", "livraison"],
                ]
            )
        ),
        "surface_m2": extraire_nombre_technique(
            valeur_champ_par_mots_cles(champs, [["surface"], ["shon"], ["sdp"]])
        ),
        "cout_total_eur": extraire_nombre_technique(
            valeur_champ_par_mots_cles(
                champs, [["cout", "total"], ["montant", "travaux"], ["cout", "travaux"]]
            )
        ),
        "volume_lot_bois_m3": extraire_nombre_technique(
            valeur_champ_par_mots_cles(
                champs, [["volume", "lot", "bois"], ["volume", "bois"]]
            )
        ),
        "cout_lot_bois_eur": extraire_nombre_technique(
            valeur_champ_par_mots_cles(
                champs, [["cout", "lot", "bois"], ["montant", "lot", "bois"]]
            )
        ),
        "isolation_murs_mm": extraire_nombre_technique(
            valeur_champ_par_mots_cles(
                champs, [["isolation", "murs"], ["isolation", "mur"]]
            )
        ),
        "approche_environnementale": valeur_champ_par_mots_cles(
            champs, [["approche", "environnementale"], ["demarche", "environnementale"]]
        ),
        "chauffage_bois": valeur_champ_par_mots_cles(champs, [["chauffage", "bois"]]),
        "appareil_chauffage_bois": valeur_champ_par_mots_cles(
            champs,
            [["appareil", "chauffage", "bois"], ["equipement", "chauffage", "bois"]],
        ),
    }

    return {colonne: ligne.get(colonne, "") for colonne in COLONNES_PROJETS_TECHNIQUES}


def valeur_ligne_tableau(ligne, noms_possibles):
    index_normalise = {
        normaliser_cle_technique(cle): valeur for cle, valeur in ligne.items()
    }

    for nom in noms_possibles:
        cle = normaliser_cle_technique(nom)
        if cle in index_normalise:
            return nettoyer_texte(index_normalise[cle])

    for cle, valeur in index_normalise.items():
        for nom in noms_possibles:
            nom_normalise = normaliser_cle_technique(nom)
            if nom_normalise and nom_normalise in cle:
                return nettoyer_texte(valeur)

    return ""


def extraire_produits_techniques(resultat):
    lignes = []

    for ligne_tableau in resultat.get("tableaux", []):
        produit = valeur_ligne_tableau(
            ligne_tableau, ["produit", "produits", "element", "ouvrage"]
        )
        essence = valeur_ligne_tableau(ligne_tableau, ["essence", "essences"])
        provenance = valeur_ligne_tableau(ligne_tableau, ["provenance", "origine"])
        if not any([produit, essence, provenance]):
            continue

        lignes.append(
            {
                "id_projet": resultat.get("numero_projet", ""),
                "categorie": nettoyer_texte(ligne_tableau.get("rubrique", "")),
                "sous_categorie": nettoyer_texte(
                    ligne_tableau.get("sous_rubrique", "")
                ),
                "produit": produit,
                "essence": essence,
                "provenance": provenance,
            }
        )

    return lignes


def sauvegarder_resultats(projets_techniques_data, produits_techniques_data):
    pd.DataFrame(projets_techniques_data, columns=COLONNES_PROJETS_TECHNIQUES).to_csv(
        FICHIER_PROJETS_TECHNIQUES, index=False, encoding="utf-8-sig"
    )

    pd.DataFrame(produits_techniques_data, columns=COLONNES_PRODUITS_TECHNIQUES).to_csv(
        FICHIER_PRODUITS_TECHNIQUES, index=False, encoding="utf-8-sig"
    )


In [ ]:

# Lecture du fichier centralisé des liens projets
df_projets = pd.read_csv(FICHIER_PROJETS, dtype=str)

if COLONNE_LIEN not in df_projets.columns:

    raise ValueError(
        f"La colonne '{COLONNE_LIEN}' n'existe pas "
        f"dans le fichier {FICHIER_PROJETS}.\n"
        f"Colonnes disponibles : "
        f"{list(df_projets.columns)}"
    )


liens_projets = df_projets[COLONNE_LIEN].dropna().astype(str).str.strip()

liens_projets = liens_projets[liens_projets != ""]

liens_projets = liens_projets.drop_duplicates().tolist()


print("Nombre total de projets :", len(liens_projets))


In [ ]:

# Listes de resultats
projets_techniques_data = []
produits_techniques_data = []
erreurs = []


# Boucle principale


for position, url in enumerate(liens_projets, start=1):

    print(f"[{position}/{len(liens_projets)}] " f"Lecture de : {url}")

    try:

        soup = charger_soup_projet_technique(url)

        resultat = extraire_tab2(soup, url)

        projets_techniques_data.append(extraire_projet_technique(resultat))

        produits_techniques_data.extend(extraire_produits_techniques(resultat))

        if not resultat["texte_complet"]:

            print("  Aucun div id='tab-2' trouve.")

    except Exception as erreur:

        print("  Erreur :", erreur)

        erreurs.append({"position": position, "lien": url, "erreur": str(erreur)})

    if position % SAUVEGARDE_TOUS_LES == 0:

        sauvegarder_resultats(
            projets_techniques_data=projets_techniques_data,
            produits_techniques_data=produits_techniques_data,
        )

        print("  Sauvegarde intermediaire terminee.")


# Sauvegarde finale
sauvegarder_resultats(
    projets_techniques_data=projets_techniques_data,
    produits_techniques_data=produits_techniques_data,
)


# Resume
print("\nExtraction terminee.")

print("Nombre de projets traites :", len(projets_techniques_data))

print("Nombre de produits extraits :", len(produits_techniques_data))

print("Nombre d'erreurs :", len(erreurs))

print("\nFichiers crees :")

print("-", FICHIER_PROJETS_TECHNIQUES)

print("-", FICHIER_PRODUITS_TECHNIQUES)


## 4.2 Filtrer les produits techniques avec essence et provenance


In [ ]:
# 1. Paramètres

fichier_produits_techniques = "projets_produits_techniques.csv"
fichier_produits_filtres = "projets_produits_techniques_essence_provenance_completes.csv"

col_essence = "essence"
col_provenance = "provenance"


# 2. Lire le fichier des produits techniques

df_produits = pd.read_csv(fichier_produits_techniques, encoding="utf-8-sig", dtype=str)

colonnes_manquantes = [
    col for col in [col_essence, col_provenance] if col not in df_produits.columns
]

if colonnes_manquantes:
    raise ValueError(
        f"Colonnes manquantes : {colonnes_manquantes}. "
        f"Colonnes disponibles : {list(df_produits.columns)}"
    )


# 3. Supprimer les lignes sans essence ou sans provenance

masque_lignes_completes = (
    df_produits[col_essence].notna()
    & df_produits[col_provenance].notna()
    & (df_produits[col_essence].astype(str).str.strip() != "")
    & (df_produits[col_provenance].astype(str).str.strip() != "")
)

df_produits_filtres = df_produits[masque_lignes_completes].copy()


# 4. Sauvegarder le nouveau CSV

df_produits_filtres.to_csv(
    fichier_produits_filtres,
    index=False,
    encoding="utf-8-sig",
)

print("Nombre de lignes originales :", len(df_produits))
print("Nombre de lignes conservées :", len(df_produits_filtres))
print("Nombre de lignes supprimées :", len(df_produits) - len(df_produits_filtres))
print("Fichier créé :")
print("-", fichier_produits_filtres)
